# 02. Спектральные характеристики ЭЭГ при ЧМТ и в контрольной группе

## Назначение ноутбука

Данный ноутбук рассчитывает спектральные характеристики ЭЭГ для записей, прошедших этап предобработки и контроля качества.

Входом является таблица `analysis_ready_inventory_cleaned.csv`, сформированная в первом ноутбуке `01_prepare_eeg_dataset.ipynb`. Таблица содержит пути к очищенным MNE Epochs-файлам, приведённым к единому набору каналов.

В рамках ноутбука рассчитываются:

1. спектральная плотность мощности PSD;
2. абсолютная и относительная мощность диапазонов δ, θ, α, β, γ;
3. индексы замедления;
4. частоты спектрального края SEF50 и SEF95;
5. спектральная энтропия;
6. spectral flatness;
7. alpha peak frequency и alpha peak amplitude;
8. параметры апериодической 1/f-компоненты;
9. residual spectrum после вычитания апериодического фона;
10. итоговая таблица спектральных признаков на уровне записи.

Основной выходной файл:

```text
spectral_features_record_level.csv
```

## Связь с этапом предобработки

Первый ноутбук выполняет:

```text
сырые .mat / EDF-файлы
        ↓
загрузка и стандартизация
        ↓
bad channels + ICA/ICLabel + AutoReject
        ↓
унификация каналов
        ↓
финальный QC
        ↓
analysis_ready_inventory_cleaned.csv
```

Текущий ноутбук не работает с сырыми файлами. Он использует только подготовленные MNE Epochs-файлы и рассчитывает количественные спектральные признаки.



# Подготовка окружения

In [ ]:
# @title 0.1. Установка зависимостей

!pip -q install mne scipy pandas numpy matplotlib tqdm scikit-learn statsmodels

In [ ]:
# @title 0.2. Импорты, стиль графиков и базовые настройки

import gc
import json
import logging
import math
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.integrate import trapezoid
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm

import mne

warnings.filterwarnings("ignore")

# Единые цвета групп во всех ноутбуках проекта.
COL_TBI = "#9B1B30"
COL_CONTROL = "#1F3F7A"

GROUP_COLORS = {
    "TBI": COL_TBI,
    "Control": COL_CONTROL,
}

GROUP_LABELS_RU = {
    "TBI": "ЧМТ",
    "Control": "Контроль",
}

plt.rcParams.update(
    {
        "figure.figsize": (12, 7),
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.family": "serif",
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
    }
)


def save_figure(fig, output_path: Path) -> None:
    """
    Сохраняет рисунок с аккуратными отступами.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Объект рисунка.
    output_path : Path
        Путь для сохранения изображения.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches="tight", dpi=300)
    print(f"Рисунок сохранён: {output_path}")


def save_table(df: pd.DataFrame, output_path: Path) -> None:
    """
    Сохраняет таблицу в CSV-файл.

    Parameters
    ----------
    df : pd.DataFrame
        Таблица для сохранения.
    output_path : Path
        Путь для сохранения CSV-файла.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Таблица сохранена: {output_path}")

In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


#1. Конфигурация спектрального анализа

В этом разделе задаются пути, частотные диапазоны, ROI и параметры расчёта PSD.

Если используется второй вариант предобработки с ICA/ICLabel/AutoReject, рекомендуется хранить результаты в отдельной рабочей папке:

```text
/work-3
```



Перед запуском укажите локальный путь к папке `work-3`, содержащей очищенный файл `analysis_ready_inventory_cleaned.csv` и подготовленные EEG epochs.

In [ ]:
# @title 1.1. Конфигурация спектрального анализа

CONFIG = {
    # ------------------------------------------------------------------
    # Пути
    # ------------------------------------------------------------------
    # Рабочая папка второго варианта обработки.
    "work_dir": Path("/path/to/work-3"),

    # Inventory после очистки, унификации каналов и финального QC.
    "analysis_ready_inventory": (
        Path("/path/to/work-3")
        / "output"
        / "tables"
        / "analysis_ready_inventory_cleaned.csv"
    ),

    # ------------------------------------------------------------------
    # Основной частотный диапазон
    # ------------------------------------------------------------------
    "fmin": 1.0,
    "fmax": 45.0,

    # ------------------------------------------------------------------
    # PSD
    # ------------------------------------------------------------------
    "psd_method": "multitaper",
    "psd_bandwidth": 2.0,
    "psd_adaptive": True,
    "psd_low_bias": True,
    "psd_normalization": "full",

    # ------------------------------------------------------------------
    # Сопоставимость числа эпох
    # ------------------------------------------------------------------
    # Если запись содержит больше эпох, берётся случайная подвыборка.
    "max_epochs_per_record": 60,
    "epoch_random_state": 97,

    # ------------------------------------------------------------------
    # Частотные диапазоны
    # ------------------------------------------------------------------
    "bands": {
        "delta": (1.0, 4.0),
        "theta": (4.0, 8.0),
        "alpha": (8.0, 13.0),
        "beta": (13.0, 30.0),
        "gamma": (30.0, 45.0),
    },

    # ------------------------------------------------------------------
    # ROI
    # ------------------------------------------------------------------
    "roi_channels": {
        "frontal": [
            "Fp1", "Fp2", "AF3", "AF4", "AF7", "AF8",
            "F1", "F2", "F3", "F4", "F5", "F6", "F7", "F8", "Fz",
        ],
        "central": [
            "C1", "C2", "C3", "C4", "C5", "C6", "Cz",
            "FC1", "FC2", "FC3", "FC4", "FC5", "FC6",
            "CP1", "CP2", "CP3", "CP4", "CP5", "CP6", "CPz",
        ],
        "temporal": [
            "T7", "T8", "FT7", "FT8", "TP7", "TP8",
        ],
        "parietal": [
            "P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8", "Pz",
            "PO3", "PO4", "PO7", "PO8", "POz",
        ],
        "occipital": [
            "O1", "O2", "Oz",
        ],
    },

    # ------------------------------------------------------------------
    # Апериодическая компонента
    # ------------------------------------------------------------------
    # Диапазон частот для линейной аппроксимации log-log PSD.
    "aperiodic_fit_range": (1.0, 45.0),

    # ------------------------------------------------------------------
    # Статистика
    # ------------------------------------------------------------------
    "alpha": 0.05,
    "n_permutations": 5000,
    "random_state": 97,
}

OUTPUT_DIR = CONFIG["work_dir"] / "output_spectral_features"

OUT = {
    "tables": OUTPUT_DIR / "tables",
    "figures": OUTPUT_DIR / "figures",
    "logs": OUTPUT_DIR / "logs",
    "cache": OUTPUT_DIR / "cache",
}

for path in OUT.values():
    path.mkdir(parents=True, exist_ok=True)

print("Конфигурация спектрального анализа задана.")
print(f"Рабочая папка: {OUTPUT_DIR}")

In [ ]:
# @title 1.2. Проверка CONFIG и analysis-ready inventory

def validate_config(config: dict) -> None:
    """
    Проверяет конфигурацию спектрального анализа.

    Parameters
    ----------
    config : dict
        Словарь CONFIG.
    """
    required_keys = [
        "work_dir",
        "analysis_ready_inventory",
        "fmin",
        "fmax",
        "bands",
        "roi_channels",
        "max_epochs_per_record",
        "epoch_random_state",
    ]

    missing_keys = [
        key for key in required_keys
        if key not in config
    ]

    if missing_keys:
        raise KeyError(f"В CONFIG отсутствуют ключи: {missing_keys}")

    inventory_path = Path(config["analysis_ready_inventory"])

    if not inventory_path.exists():
        raise FileNotFoundError(
            "Не найден analysis-ready inventory: "
            f"{inventory_path}"
        )

    if config["fmin"] >= config["fmax"]:
        raise ValueError("fmin должен быть меньше fmax.")

    if config["max_epochs_per_record"] <= 0:
        raise ValueError("max_epochs_per_record должен быть положительным.")

    if len(config["bands"]) == 0:
        raise ValueError("Словарь bands не должен быть пустым.")

    if len(config["roi_channels"]) == 0:
        raise ValueError("Словарь roi_channels не должен быть пустым.")


validate_config(CONFIG)

inventory_check = pd.read_csv(CONFIG["analysis_ready_inventory"])

print("CONFIG спектрального анализа успешно проверен.")
print("Analysis-ready inventory найден:")
print(CONFIG["analysis_ready_inventory"])
print(f"Записей: {len(inventory_check)}")

display(
    inventory_check
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
        median_epochs=("n_epochs", "median"),
        n_common_channels=("n_common_channels", "median"),
    )
    .reset_index()
)

In [ ]:
# @title 1.2. Настройка логирования

def setup_logger(output_dir: Path) -> logging.Logger:
    """
    Настраивает logger для текущего ноутбука.

    Parameters
    ----------
    output_dir : Path
        Папка для сохранения логов.

    Returns
    -------
    logging.Logger
        Настроенный logger.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger("spectral_features")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    log_path = output_dir / (
        "spectral_features_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    )

    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    file_handler.setLevel(logging.INFO)

    stream_handler = logging.StreamHandler()
    stream_handler.setLevel(logging.INFO)

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler.setFormatter(formatter)
    stream_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)

    logger.info("Логирование спектрального анализа запущено.")
    logger.info("Файл лога: %s", log_path)

    return logger


logger = setup_logger(OUT["logs"])

In [ ]:
# @title 1.3. Проверка конфигурации

def validate_config(config: dict) -> None:
    """
    Проверяет корректность конфигурации спектрального анализа.

    Parameters
    ----------
    config : dict
        Словарь конфигурации.
    """
    inventory_path = config["analysis_ready_inventory"]

    if not inventory_path.exists():
        raise FileNotFoundError(
            "Не найден analysis-ready inventory: "
            f"{inventory_path}"
        )

    if config["fmin"] <= 0:
        raise ValueError("fmin должен быть положительным числом.")

    if config["fmin"] >= config["fmax"]:
        raise ValueError("fmin должен быть меньше fmax.")

    if config["max_epochs_per_record"] <= 0:
        raise ValueError("max_epochs_per_record должен быть положительным.")

    for band_name, (band_min, band_max) in config["bands"].items():
        if band_min >= band_max:
            raise ValueError(
                f"Некорректный диапазон {band_name}: "
                f"{band_min}–{band_max}"
            )


validate_config(CONFIG)

logger.info("CONFIG спектрального анализа успешно проверен.")
print("CONFIG корректен. Можно переходить к загрузке inventory.")

# 2. Загрузка подготовленного inventory

На этом этапе загружается таблица `analysis_ready_inventory_cleaned.csv`.

Она должна содержать:

- группу;
- идентификатор субъекта;
- идентификатор записи;
- путь к очищенному MNE Epochs-файлу;
- число эпох;
- число каналов;
- частоту дискретизации;
- длительность эпох.

Эта таблица является единственной точкой входа для спектрального анализа.

In [ ]:
# @title 2.1. Загрузка analysis-ready inventory

analysis_inventory = pd.read_csv(CONFIG["analysis_ready_inventory"])

required_columns = [
    "group",
    "subject_id",
    "record_id",
    "common_epochs_path",
    "n_epochs",
    "n_common_channels",
    "sfreq_hz",
    "epoch_len_sec",
]

missing_columns = [
    column for column in required_columns
    if column not in analysis_inventory.columns
]

if missing_columns:
    raise ValueError(
        "В inventory отсутствуют обязательные колонки: "
        f"{missing_columns}"
    )

print("Inventory загружен.")
print(f"Число записей: {len(analysis_inventory)}")
print(f"Число субъектов: {analysis_inventory['subject_id'].nunique()}")

display(
    analysis_inventory
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
        median_epochs=("n_epochs", "median"),
        n_channels=("n_common_channels", "median"),
        sfreq_hz=("sfreq_hz", "median"),
        epoch_len_sec=("epoch_len_sec", "median"),
    )
    .reset_index()
)

In [ ]:
# @title 2.2. Проверка доступности MNE Epochs-файлов

analysis_inventory["epochs_file_exists"] = analysis_inventory[
    "common_epochs_path"
].apply(lambda path: Path(path).exists())

missing_epochs = analysis_inventory[
    ~analysis_inventory["epochs_file_exists"]
].copy()

print("Проверка путей к MNE Epochs-файлам")
print("-" * 70)
print(f"Всего записей: {len(analysis_inventory)}")
print(
    "Файлы найдены: "
    f"{analysis_inventory['epochs_file_exists'].sum()}"
)
print(f"Файлы не найдены: {len(missing_epochs)}")

if len(missing_epochs) > 0:
    display(
        missing_epochs[
            [
                "group",
                "subject_id",
                "record_id",
                "common_epochs_path",
            ]
        ].head(20)
    )

    raise FileNotFoundError(
        "Часть MNE Epochs-файлов не найдена. "
        "Проверьте пути в common_epochs_path."
    )

logger.info("Все MNE Epochs-файлы доступны.")

In [ ]:
# @title 2.3. Визуальная сводка входных данных

summary_for_plot = (
    analysis_inventory
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
        n_epochs=("n_epochs", "sum"),
        median_epochs=("n_epochs", "median"),
    )
    .reset_index()
)

summary_for_plot["group_ru"] = summary_for_plot["group"].map(
    GROUP_LABELS_RU
)
summary_for_plot["color"] = summary_for_plot["group"].map(GROUP_COLORS)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [
    ("n_subjects", "Число субъектов"),
    ("n_records", "Число записей"),
    ("n_epochs", "Число эпох"),
]

for ax, (column, title) in zip(axes, metrics):
    bars = ax.bar(
        summary_for_plot["group_ru"],
        summary_for_plot[column],
        color=summary_for_plot["color"],
    )

    ax.set_title(title)
    ax.set_ylabel("Количество")

    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"{int(height)}",
            ha="center",
            va="bottom",
        )

fig.suptitle(
    "Состав данных на входе спектрального анализа",
    fontsize=17,
    y=1.03,
)

fig.tight_layout()

figure_path = OUT["figures"] / "input_inventory_summary.png"
save_figure(fig, figure_path)

plt.show()

### Результат проверки входных данных

На вход спектрального анализа поступают только записи, прошедшие предобработку, очистку артефактов, унификацию каналов и финальный контроль качества.

Дальнейшие признаки рассчитываются на уровне записи. Если у одного субъекта доступно несколько записей, это учитывается на этапе статистики и машинного обучения: разбиения должны выполняться на уровне субъекта, чтобы исключить утечку информации между train/test.

# 3. Общие функции загрузки эпох и выбора подвыборки

Перед расчётом PSD задаются универсальные функции:

1. загрузка MNE Epochs-файла;
2. выбор фиксированного числа эпох из записи;
3. выбор каналов ROI;
4. безопасное получение данных для дальнейших расчётов.

Подвыборка эпох используется только в том случае, если запись содержит больше заданного максимального числа эпох.

In [ ]:
# @title 3.1. Загрузка Epochs и подвыборка эпох

def load_epochs_file(epochs_path: str | Path) -> mne.Epochs:
    """
    Загружает MNE Epochs-файл.

    Parameters
    ----------
    epochs_path : str | Path
        Путь к FIF-файлу с эпохами.

    Returns
    -------
    mne.Epochs
        Загруженный объект MNE Epochs.
    """
    epochs_path = Path(epochs_path)

    if not epochs_path.exists():
        raise FileNotFoundError(f"Файл эпох не найден: {epochs_path}")

    epochs = mne.read_epochs(
        epochs_path,
        preload=True,
        verbose=False,
    )

    return epochs


def select_epochs_subset(
    epochs: mne.Epochs,
    max_epochs: int,
    random_state: int,
) -> mne.Epochs:
    """
    Выбирает подвыборку эпох фиксированного максимального размера.

    Parameters
    ----------
    epochs : mne.Epochs
        Исходные эпохи.
    max_epochs : int
        Максимальное число эпох.
    random_state : int
        Seed генератора случайных чисел.

    Returns
    -------
    mne.Epochs
        Epochs после подвыборки.
    """
    if len(epochs) <= max_epochs:
        return epochs.copy()

    rng = np.random.default_rng(random_state)

    selected_indices = rng.choice(
        np.arange(len(epochs)),
        size=max_epochs,
        replace=False,
    )

    selected_indices = np.sort(selected_indices)

    return epochs[selected_indices].copy()


def get_available_roi_channels(
    epochs: mne.Epochs,
    roi_channels: dict[str, list[str]],
) -> dict[str, list[str]]:
    """
    Возвращает доступные каналы для каждого ROI.

    Parameters
    ----------
    epochs : mne.Epochs
        Объект эпох.
    roi_channels : dict[str, list[str]]
        Словарь ROI и соответствующих каналов.

    Returns
    -------
    dict[str, list[str]]
        Словарь ROI с каналами, присутствующими в записи.
    """
    available_channels = set(epochs.ch_names)

    available_roi_channels = {}

    for roi_name, channels in roi_channels.items():
        selected_channels = [
            channel for channel in channels
            if channel in available_channels
        ]

        available_roi_channels[roi_name] = selected_channels

    return available_roi_channels

In [ ]:
# @title 3.2. Тестовая загрузка одной записи

test_row = analysis_inventory.iloc[0]

print("Тестовая запись")
print("-" * 70)
print(f"Группа: {test_row['group']}")
print(f"Субъект: {test_row['subject_id']}")
print(f"Запись: {test_row['record_id']}")

test_epochs = load_epochs_file(test_row["common_epochs_path"])

test_epochs_subset = select_epochs_subset(
    epochs=test_epochs,
    max_epochs=CONFIG["max_epochs_per_record"],
    random_state=CONFIG["epoch_random_state"],
)

available_roi_channels = get_available_roi_channels(
    epochs=test_epochs_subset,
    roi_channels=CONFIG["roi_channels"],
)

print("\nПараметры Epochs:")
print(f"Число эпох исходно: {len(test_epochs)}")
print(f"Число эпох после подвыборки: {len(test_epochs_subset)}")
print(f"Число каналов: {len(test_epochs_subset.ch_names)}")
print(f"Частота дискретизации: {test_epochs_subset.info['sfreq']} Гц")
print(
    "Длительность эпохи: "
    f"{test_epochs_subset.times[-1] - test_epochs_subset.times[0]:.3f} с"
)

print("\nДоступные каналы по ROI:")
for roi_name, channels in available_roi_channels.items():
    print(f"{roi_name}: {len(channels)} каналов")

del test_epochs
del test_epochs_subset
gc.collect()

# Спектральные характеристики

# 4. Расчёт спектральной плотности мощности PSD

На этом этапе для каждой записи рассчитывается PSD методом multitaper.

Для каждой записи:

1. загружается MNE Epochs-файл;
2. при необходимости выбирается подвыборка эпох;
3. рассчитывается PSD в диапазоне 1–45 Гц;
4. PSD усредняется по эпохам;
5. сохраняется глобальная PSD, PSD по каналам и PSD по ROI.

PSD является основой для дальнейшего расчёта диапазонной мощности, индексов замедления, SEF, entropy, alpha peak и апериодической компоненты.

In [ ]:
# @title 4.1. Функции расчёта PSD

def compute_record_psd(
    epochs: mne.Epochs,
    config: dict,
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Рассчитывает PSD для одной записи.

    Parameters
    ----------
    epochs : mne.Epochs
        Эпохированные ЭЭГ-данные.
    config : dict
        Конфигурация анализа.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, list[str]]
        PSD shape = (n_channels, n_freqs), частоты и список каналов.
    """
    epochs = select_epochs_subset(
        epochs=epochs,
        max_epochs=config["max_epochs_per_record"],
        random_state=config["epoch_random_state"],
    )

    spectrum = epochs.compute_psd(
        method=config["psd_method"],
        fmin=config["fmin"],
        fmax=config["fmax"],
        bandwidth=config["psd_bandwidth"],
        adaptive=config["psd_adaptive"],
        low_bias=config["psd_low_bias"],
        normalization=config["psd_normalization"],
        picks="eeg",
        verbose=False,
    )

    psd = spectrum.get_data()
    freqs = spectrum.freqs

    # Усредняем PSD по эпохам. Остаётся channel x frequency.
    psd_mean = np.nanmean(psd, axis=0)

    return psd_mean, freqs, list(spectrum.ch_names)


def aggregate_psd_by_roi(
    psd: np.ndarray,
    ch_names: list[str],
    roi_channels: dict[str, list[str]],
) -> dict[str, np.ndarray]:
    """
    Агрегирует PSD по ROI.

    Parameters
    ----------
    psd : np.ndarray
        PSD shape = (n_channels, n_freqs).
    ch_names : list[str]
        Список каналов PSD.
    roi_channels : dict[str, list[str]]
        Словарь ROI.

    Returns
    -------
    dict[str, np.ndarray]
        PSD по ROI.
    """
    ch_to_idx = {
        channel: index for index, channel in enumerate(ch_names)
    }

    roi_psd = {}

    for roi_name, channels in roi_channels.items():
        indices = [
            ch_to_idx[channel]
            for channel in channels
            if channel in ch_to_idx
        ]

        if len(indices) == 0:
            roi_psd[roi_name] = np.full(psd.shape[1], np.nan)
            continue

        roi_psd[roi_name] = np.nanmean(psd[indices, :], axis=0)

    return roi_psd

In [ ]:
# @title 4.2. Массовый расчёт PSD по записям

psd_records = []
roi_psd_records = []
psd_cache_path = OUT["cache"] / "psd_record_level.npz"

all_global_psd = []
all_freqs = None

for _, row in tqdm(
    analysis_inventory.iterrows(),
    total=len(analysis_inventory),
    desc="Расчёт PSD",
):
    try:
        epochs = load_epochs_file(row["common_epochs_path"])

        psd, freqs, ch_names = compute_record_psd(
            epochs=epochs,
            config=CONFIG,
        )

        if all_freqs is None:
            all_freqs = freqs

        global_psd = np.nanmean(psd, axis=0)
        all_global_psd.append(global_psd)

        base_record = {
            "group": row["group"],
            "subject_id": row["subject_id"],
            "record_id": row["record_id"],
            "n_epochs_original": row["n_epochs"],
            "n_epochs_used": min(
                row["n_epochs"],
                CONFIG["max_epochs_per_record"],
            ),
            "n_channels": len(ch_names),
        }

        for freq, value in zip(freqs, global_psd):
            psd_records.append(
                {
                    **base_record,
                    "level": "global",
                    "region": "global",
                    "freq_hz": float(freq),
                    "psd": float(value),
                }
            )

        roi_psd = aggregate_psd_by_roi(
            psd=psd,
            ch_names=ch_names,
            roi_channels=CONFIG["roi_channels"],
        )

        for roi_name, roi_values in roi_psd.items():
            for freq, value in zip(freqs, roi_values):
                roi_psd_records.append(
                    {
                        **base_record,
                        "level": "roi",
                        "region": roi_name,
                        "freq_hz": float(freq),
                        "psd": float(value),
                    }
                )

        del epochs
        gc.collect()

    except Exception as error:
        logger.exception(
            "Ошибка PSD для записи %s: %s",
            row.get("record_id"),
            error,
        )

psd_global_long = pd.DataFrame(psd_records)
psd_roi_long = pd.DataFrame(roi_psd_records)

save_table(
    psd_global_long,
    OUT["tables"] / "psd_global_long.csv",
)

save_table(
    psd_roi_long,
    OUT["tables"] / "psd_roi_long.csv",
)

np.savez_compressed(
    psd_cache_path,
    freqs=all_freqs,
    global_psd=np.asarray(all_global_psd),
)

print("Расчёт PSD завершён.")
print(f"Глобальная PSD: {psd_global_long.shape}")
print(f"ROI PSD: {psd_roi_long.shape}")
print(f"Кэш PSD сохранён: {psd_cache_path}")

In [ ]:
# @title 4.3. Визуализация средней глобальной PSD

fig, ax = plt.subplots(figsize=(11, 6))

for group_name, color in GROUP_COLORS.items():
    group_psd = psd_global_long[
        psd_global_long["group"] == group_name
    ]

    summary = (
        group_psd
        .groupby("freq_hz")
        .agg(
            mean_psd=("psd", "mean"),
            sem_psd=("psd", "sem"),
        )
        .reset_index()
    )

    ax.plot(
        summary["freq_hz"],
        summary["mean_psd"],
        color=color,
        linewidth=2,
        label=GROUP_LABELS_RU[group_name],
    )

    ax.fill_between(
        summary["freq_hz"],
        summary["mean_psd"] - summary["sem_psd"],
        summary["mean_psd"] + summary["sem_psd"],
        color=color,
        alpha=0.15,
    )

ax.set_title("Средняя глобальная PSD")
ax.set_xlabel("Частота, Гц")
ax.set_ylabel("PSD")
ax.set_xlim(CONFIG["fmin"], CONFIG["fmax"])
ax.legend(frameon=False)

figure_path = OUT["figures"] / "global_psd_group_mean.png"
save_figure(fig, figure_path)

plt.show()

### Результат расчёта PSD

На этом этапе для каждой записи была рассчитана PSD в диапазоне 1–45 Гц. Получены две основные таблицы:

- `psd_global_long.csv` — глобальный спектр, усреднённый по всем каналам;
- `psd_roi_long.csv` — спектры, усреднённые по ROI.

Эти таблицы используются для расчёта спектральных признаков.

# 5. Диапазонная мощность и индексы замедления

На основе PSD рассчитываются абсолютная и относительная мощность в стандартных частотных диапазонах:

| Диапазон | Частоты |
|---|---:|
| δ | 1–4 Гц |
| θ | 4–8 Гц |
| α | 8–13 Гц |
| β | 13–30 Гц |
| γ | 30–45 Гц |

Также рассчитываются индексы замедления:

- `(δ + θ) / α`;
- `(δ + θ) / β`;
- `(δ + θ) / (α + β)`;
- `log((δ + θ) / (α + β))`.

Эти индексы используются как компактные показатели смещения спектра в сторону медленных частот.

In [ ]:
# @title 5.1. Функции расчёта bandpower и индексов

def integrate_band_power(
    freqs: np.ndarray,
    psd_values: np.ndarray,
    band: tuple[float, float],
) -> float:
    """
    Интегрирует PSD внутри частотного диапазона.

    Parameters
    ----------
    freqs : np.ndarray
        Частоты.
    psd_values : np.ndarray
        PSD-значения.
    band : tuple[float, float]
        Частотный диапазон.

    Returns
    -------
    float
        Мощность в диапазоне.
    """
    band_min, band_max = band
    band_mask = (freqs >= band_min) & (freqs < band_max)

    if np.sum(band_mask) < 2:
        return np.nan

    return float(trapezoid(psd_values[band_mask], freqs[band_mask]))


def compute_bandpower_features(
    psd_long: pd.DataFrame,
    config: dict,
) -> pd.DataFrame:
    """
    Рассчитывает absolute и relative bandpower.

    Parameters
    ----------
    psd_long : pd.DataFrame
        Long-таблица PSD.
    config : dict
        Конфигурация анализа.

    Returns
    -------
    pd.DataFrame
        Таблица bandpower на уровне записи и региона.
    """
    records = []

    group_columns = [
        "group",
        "subject_id",
        "record_id",
        "level",
        "region",
    ]

    for keys, sub_df in psd_long.groupby(group_columns):
        freqs = sub_df["freq_hz"].to_numpy()
        psd_values = sub_df["psd"].to_numpy()

        total_power = integrate_band_power(
            freqs=freqs,
            psd_values=psd_values,
            band=(config["fmin"], config["fmax"]),
        )

        record = dict(zip(group_columns, keys))
        record["total_power_1_45"] = total_power

        for band_name, band_range in config["bands"].items():
            abs_power = integrate_band_power(
                freqs=freqs,
                psd_values=psd_values,
                band=band_range,
            )

            record[f"abs_power_{band_name}"] = abs_power
            record[f"rel_power_{band_name}"] = (
                abs_power / total_power
                if total_power and np.isfinite(total_power)
                else np.nan
            )

        records.append(record)

    return pd.DataFrame(records)


def add_slow_fast_indices(df: pd.DataFrame) -> pd.DataFrame:
    """
    Добавляет индексы соотношения медленной и быстрой активности.

    Parameters
    ----------
    df : pd.DataFrame
        Таблица bandpower.

    Returns
    -------
    pd.DataFrame
        Таблица с индексами.
    """
    df = df.copy()

    slow = df["rel_power_delta"] + df["rel_power_theta"]
    alpha = df["rel_power_alpha"]
    beta = df["rel_power_beta"]

    eps = 1e-12

    df["idx_slow_alpha"] = slow / (alpha + eps)
    df["idx_slow_beta"] = slow / (beta + eps)
    df["idx_slow_alpha_beta"] = slow / (alpha + beta + eps)
    df["idx_log_slow_alpha_beta"] = np.log(
        (slow + eps) / (alpha + beta + eps)
    )

    return df

In [ ]:
# @title 5.2. Расчёт bandpower и индексов

bandpower_global = compute_bandpower_features(
    psd_long=psd_global_long,
    config=CONFIG,
)

bandpower_roi = compute_bandpower_features(
    psd_long=psd_roi_long,
    config=CONFIG,
)

bandpower_global = add_slow_fast_indices(bandpower_global)
bandpower_roi = add_slow_fast_indices(bandpower_roi)

save_table(
    bandpower_global,
    OUT["tables"] / "bandpower_global.csv",
)

save_table(
    bandpower_roi,
    OUT["tables"] / "bandpower_roi.csv",
)

print("Bandpower рассчитан.")
display(bandpower_roi.head())

In [ ]:
# @title 5.3. Тепловая карта относительной мощности по ROI

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import TwoSlopeNorm

# ---------------------------------------------------------------------
# Порядок отображения
# ---------------------------------------------------------------------
bands_order = ["delta", "theta", "alpha", "beta", "gamma"]
roi_order = ["frontal", "central", "temporal", "parietal", "occipital"]

band_labels = {
    "delta": "δ (1–4 Гц)",
    "theta": "θ (4–8 Гц)",
    "alpha": "α (8–13 Гц)",
    "beta": "β (13–30 Гц)",
    "gamma": "γ (30–45 Гц)",
}

roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

group_labels = {
    "TBI": "ЧМТ",
    "Control": "Контроль",
    "tbi": "ЧМТ",
    "control": "Контроль",
}

# ---------------------------------------------------------------------
# Подготовка таблицы для heatmap
# ---------------------------------------------------------------------
bandpower_plot = bandpower_roi.copy()

# На случай, если группы в таблице записаны как tbi/control.
if set(bandpower_plot["group"].dropna().unique()) >= {"tbi", "control"}:
    group_tbi = "tbi"
    group_control = "control"
else:
    group_tbi = "TBI"
    group_control = "Control"

heatmap_records = []

for group_name in [group_tbi, group_control]:
    group_df = bandpower_plot[bandpower_plot["group"] == group_name]

    for roi_name in roi_order:
        roi_df = group_df[group_df["region"] == roi_name]

        for band_name in bands_order:
            heatmap_records.append(
                {
                    "group": group_name,
                    "roi": roi_name,
                    "band": band_name,
                    "value": roi_df[f"rel_power_{band_name}"].mean(),
                }
            )

heatmap_df = pd.DataFrame(heatmap_records)

pivot_tbi = (
    heatmap_df[heatmap_df["group"] == group_tbi]
    .pivot(index="roi", columns="band", values="value")
    .loc[roi_order, bands_order]
)

pivot_control = (
    heatmap_df[heatmap_df["group"] == group_control]
    .pivot(index="roi", columns="band", values="value")
    .loc[roi_order, bands_order]
)

pivot_diff = pivot_tbi - pivot_control

# ---------------------------------------------------------------------
# Общие пределы цвета для ЧМТ и контроля
# ---------------------------------------------------------------------
main_vmin = min(
    np.nanmin(pivot_tbi.values),
    np.nanmin(pivot_control.values),
)

main_vmax = max(
    np.nanmax(pivot_tbi.values),
    np.nanmax(pivot_control.values),
)

diff_abs = np.nanmax(np.abs(pivot_diff.values))
diff_norm = TwoSlopeNorm(vmin=-diff_abs, vcenter=0, vmax=diff_abs)

# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5.2),
    constrained_layout=True,
)

fig.suptitle(
    "Относительная мощность по ROI и диапазонам",
    fontsize=15,
    fontweight="bold",
)

panels = [
    {
        "ax": axes[0],
        "data": pivot_tbi,
        "title": "ЧМТ",
        "cmap": "Blues",
        "vmin": main_vmin,
        "vmax": main_vmax,
        "norm": None,
        "cbar_label": "Относительная мощность",
    },
    {
        "ax": axes[1],
        "data": pivot_control,
        "title": "Контроль",
        "cmap": "Blues",
        "vmin": main_vmin,
        "vmax": main_vmax,
        "norm": None,
        "cbar_label": "Относительная мощность",
    },
    {
        "ax": axes[2],
        "data": pivot_diff,
        "title": "Разность (ЧМТ − Контроль)",
        "cmap": "RdBu_r",
        "vmin": None,
        "vmax": None,
        "norm": diff_norm,
        "cbar_label": "Разность",
    },
]

for panel in panels:
    ax = panel["ax"]
    data = panel["data"]

    if panel["norm"] is None:
        im = ax.imshow(
            data.values,
            aspect="auto",
            cmap=panel["cmap"],
            vmin=panel["vmin"],
            vmax=panel["vmax"],
        )
    else:
        im = ax.imshow(
            data.values,
            aspect="auto",
            cmap=panel["cmap"],
            norm=panel["norm"],
        )

    ax.set_title(panel["title"], fontsize=12, fontweight="bold")

    ax.set_xticks(np.arange(len(bands_order)))
    ax.set_xticklabels(
        [band_labels[band] for band in bands_order],
        rotation=25,
        ha="right",
    )

    ax.set_yticks(np.arange(len(roi_order)))
    ax.set_yticklabels([roi_labels[roi] for roi in roi_order])

    # Сетка между ячейками.
    ax.set_xticks(np.arange(-0.5, len(bands_order), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(roi_order), 1), minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=1.0)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Подписи значений в ячейках.
    values = data.values
    threshold = np.nanmean(values)

    for i in range(len(roi_order)):
        for j in range(len(bands_order)):
            value = values[i, j]

            if np.isnan(value):
                label = "NA"
                text_color = "black"
            else:
                label = f"{value:.3f}"

                if panel["title"] == "Разность (ЧМТ − Контроль)":
                    text_color = (
                        "white"
                        if abs(value) > 0.55 * diff_abs
                        else "black"
                    )
                else:
                    text_color = (
                        "white"
                        if value > threshold
                        else "black"
                    )

            ax.text(
                j,
                i,
                label,
                ha="center",
                va="center",
                fontsize=9,
                color=text_color,
            )

    cbar = fig.colorbar(
        im,
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )
    cbar.set_label(panel["cbar_label"], fontsize=10)

# ---------------------------------------------------------------------
# Сохранение
# ---------------------------------------------------------------------
figure_path = OUT["figures"] / "relative_bandpower_heatmap_roi_combined.png"

save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 5.4. Распределения относительной мощности по диапазонам частот и ROI

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---------------------------------------------------------------------
# Порядок отображения и подписи
# ---------------------------------------------------------------------
bands_order = ["delta", "theta", "alpha", "beta", "gamma"]
roi_order = ["frontal", "central", "temporal", "parietal", "occipital"]

band_labels = {
    "delta": "δ (1–4 Гц)",
    "theta": "θ (4–8 Гц)",
    "alpha": "α (8–13 Гц)",
    "beta": "β (13–30 Гц)",
    "gamma": "γ (30–45 Гц)",
}

roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

# ---------------------------------------------------------------------
# Подготовка данных
# ---------------------------------------------------------------------
plot_df = bandpower_roi.copy()

plot_df["group_norm"] = plot_df["group"].map(
    {
        "TBI": "tbi",
        "tbi": "tbi",
        "ЧМТ": "tbi",
        "Control": "control",
        "control": "control",
        "Контроль": "control",
    }
)

plot_df = plot_df[plot_df["group_norm"].isin(["tbi", "control"])].copy()

# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
fig, axes = plt.subplots(
    2,
    3,
    figsize=(15.5, 8.2),
    sharex=False,
    sharey=False,
)

axes = axes.ravel()

fig.suptitle(
    "Распределения относительной мощности по диапазонам частот и ROI",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

roi_positions = np.arange(len(roi_order))
offset = 0.18
width = 0.32

for idx, band_name in enumerate(bands_order):
    ax = axes[idx]

    feature_col = f"rel_power_{band_name}"

    tbi_data = [
        plot_df.loc[
            (plot_df["group_norm"] == "tbi")
            & (plot_df["region"] == roi_name),
            feature_col,
        ].replace([np.inf, -np.inf], np.nan).dropna()
        for roi_name in roi_order
    ]

    control_data = [
        plot_df.loc[
            (plot_df["group_norm"] == "control")
            & (plot_df["region"] == roi_name),
            feature_col,
        ].replace([np.inf, -np.inf], np.nan).dropna()
        for roi_name in roi_order
    ]

    ax.boxplot(
        tbi_data,
        positions=roi_positions - offset,
        widths=width,
        patch_artist=True,
        showfliers=True,
        medianprops={
            "color": "black",
            "linewidth": 1.4,
        },
        boxprops={
            "facecolor": COL_TBI,
            "edgecolor": "black",
            "linewidth": 1.1,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.0,
        },
        capprops={
            "color": "black",
            "linewidth": 1.0,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": COL_TBI,
            "markeredgecolor": COL_TBI,
            "markersize": 2.4,
            "alpha": 0.55,
        },
    )

    ax.boxplot(
        control_data,
        positions=roi_positions + offset,
        widths=width,
        patch_artist=True,
        showfliers=True,
        medianprops={
            "color": "black",
            "linewidth": 1.4,
        },
        boxprops={
            "facecolor": COL_CONTROL,
            "edgecolor": "black",
            "linewidth": 1.1,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.0,
        },
        capprops={
            "color": "black",
            "linewidth": 1.0,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": COL_CONTROL,
            "markeredgecolor": COL_CONTROL,
            "markersize": 2.4,
            "alpha": 0.55,
        },
    )

    ax.set_title(
        band_labels[band_name],
        fontsize=12,
        fontweight="bold",
        pad=6,
    )

    ax.set_ylabel("Относительная мощность", fontsize=10.5)

    ax.set_xticks(roi_positions)
    ax.set_xticklabels(
        [roi_labels[roi] for roi in roi_order],
        rotation=22,
        ha="right",
    )

    ax.grid(axis="y", linestyle="--", alpha=0.22)
    ax.grid(axis="x", linestyle=":", alpha=0.10)

    # -------------------------------------------------------------
    # Robust-масштаб по Y: данные не меняются, только ось.
    # Это делает boxplot читаемым при отдельных выбросах.
    # -------------------------------------------------------------
    band_values = (
        plot_df[feature_col]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    y_low = max(0, np.nanpercentile(band_values, 0.5))
    y_high = np.nanpercentile(band_values, 99.2)

    if np.isfinite(y_high) and y_high > y_low:
        ax.set_ylim(y_low, y_high * 1.08)

    ax.text(
        0.98,
        0.96,
        "ось Y: до 99.2 перцентиля",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=7.8,
        color="dimgray",
    )

# Пустую шестую панель выключаем.
axes[-1].axis("off")

# ---------------------------------------------------------------------
# Общая легенда снизу
# ---------------------------------------------------------------------
legend_handles = [
    mpatches.Patch(
        facecolor=COL_TBI,
        edgecolor="black",
        label="ЧМТ",
    ),
    mpatches.Patch(
        facecolor=COL_CONTROL,
        edgecolor="black",
        label="Контроль",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 0.015),
)

# ---------------------------------------------------------------------
# Расстояния между панелями
# ---------------------------------------------------------------------
fig.subplots_adjust(
    left=0.06,
    right=0.98,
    top=0.90,
    bottom=0.13,
    wspace=0.16,
    hspace=0.34,
)

# ---------------------------------------------------------------------
# Сохранение
# ---------------------------------------------------------------------
figure_path = OUT["figures"] / "relative_bandpower_boxplots_by_band_roi.png"

save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 5.5. Индексы спектрального замедления по ROI

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---------------------------------------------------------------------
# Порядок отображения и подписи
# ---------------------------------------------------------------------
roi_order = ["frontal", "central", "temporal", "parietal", "occipital"]

roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

index_info = [
    {
        "column": "slow_alpha_ratio",
        "title": "Индекс замедления (δ + θ) / α",
        "ylabel": "Значение индекса",
        "robust_ylim": True,
    },
    {
        "column": "slow_beta_ratio",
        "title": "Индекс замедления (δ + θ) / β",
        "ylabel": "Значение индекса",
        "robust_ylim": True,
    },
    {
        "column": "log_slow_fast_ratio",
        "title": "Лог-индекс log((δ + θ) / (α + β))",
        "ylabel": "Значение индекса",
        "robust_ylim": False,
    },
]

# ---------------------------------------------------------------------
# Подготовка данных
# ---------------------------------------------------------------------
plot_df = bandpower_roi.copy()

plot_df["group_norm"] = plot_df["group"].map(
    {
        "TBI": "tbi",
        "tbi": "tbi",
        "ЧМТ": "tbi",
        "Control": "control",
        "control": "control",
        "Контроль": "control",
    }
)

plot_df = plot_df[plot_df["group_norm"].isin(["tbi", "control"])].copy()

# ---------------------------------------------------------------------
# Расчёт индексов, если они ещё не были созданы
# ---------------------------------------------------------------------
eps = 1e-12

if "slow_alpha_ratio" not in plot_df.columns:
    plot_df["slow_alpha_ratio"] = (
        plot_df["rel_power_delta"] + plot_df["rel_power_theta"]
    ) / (plot_df["rel_power_alpha"] + eps)

if "slow_beta_ratio" not in plot_df.columns:
    plot_df["slow_beta_ratio"] = (
        plot_df["rel_power_delta"] + plot_df["rel_power_theta"]
    ) / (plot_df["rel_power_beta"] + eps)

if "log_slow_fast_ratio" not in plot_df.columns:
    plot_df["log_slow_fast_ratio"] = np.log(
        (
            plot_df["rel_power_delta"] + plot_df["rel_power_theta"] + eps
        )
        / (
            plot_df["rel_power_alpha"] + plot_df["rel_power_beta"] + eps
        )
    )

# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
fig, axes = plt.subplots(
    3,
    1,
    figsize=(14, 9.2),
    sharex=True,
    constrained_layout=True,
)

fig.suptitle(
    "Индексы спектрального замедления: сравнение групп по ROI",
    fontsize=14,
    fontweight="bold",
)

roi_positions = np.arange(len(roi_order))
offset = 0.18
width = 0.32

for ax, info in zip(axes, index_info):
    feature_col = info["column"]

    tbi_data = [
        plot_df.loc[
            (plot_df["group_norm"] == "tbi")
            & (plot_df["region"] == roi_name),
            feature_col,
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        for roi_name in roi_order
    ]

    control_data = [
        plot_df.loc[
            (plot_df["group_norm"] == "control")
            & (plot_df["region"] == roi_name),
            feature_col,
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        for roi_name in roi_order
    ]

    ax.boxplot(
        tbi_data,
        positions=roi_positions - offset,
        widths=width,
        patch_artist=True,
        showfliers=True,
        medianprops={
            "color": "black",
            "linewidth": 1.4,
        },
        boxprops={
            "facecolor": COL_TBI,
            "edgecolor": "black",
            "linewidth": 1.1,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.0,
        },
        capprops={
            "color": "black",
            "linewidth": 1.0,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": COL_TBI,
            "markeredgecolor": COL_TBI,
            "markersize": 2.2,
            "alpha": 0.55,
        },
    )

    ax.boxplot(
        control_data,
        positions=roi_positions + offset,
        widths=width,
        patch_artist=True,
        showfliers=True,
        medianprops={
            "color": "black",
            "linewidth": 1.4,
        },
        boxprops={
            "facecolor": COL_CONTROL,
            "edgecolor": "black",
            "linewidth": 1.1,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.0,
        },
        capprops={
            "color": "black",
            "linewidth": 1.0,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": COL_CONTROL,
            "markeredgecolor": COL_CONTROL,
            "markersize": 2.2,
            "alpha": 0.55,
        },
    )

    ax.set_title(info["title"], fontsize=11.5, fontweight="bold", pad=6)
    ax.set_ylabel(info["ylabel"])
    ax.grid(axis="y", linestyle="--", alpha=0.25)
    ax.grid(axis="x", linestyle=":", alpha=0.12)

    # -----------------------------------------------------------------
    # Ограничиваем ось Y для читаемости, не меняя сами данные
    # -----------------------------------------------------------------
    all_values = (
        plot_df[feature_col]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if info["robust_ylim"]:
        upper = np.nanpercentile(all_values, 99)
        lower = max(0, np.nanpercentile(all_values, 0.5))

        ax.set_ylim(lower, upper * 1.08)

        ax.text(
            0.995,
            0.96,
            "ось Y ограничена по 99-му перцентилю",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=8.5,
            color="dimgray",
        )
    else:
        lower = np.nanpercentile(all_values, 1)
        upper = np.nanpercentile(all_values, 99)

        margin = 0.12 * (upper - lower)
        ax.set_ylim(lower - margin, upper + margin)

# ---------------------------------------------------------------------
# Ось X
# ---------------------------------------------------------------------
axes[-1].set_xticks(roi_positions)
axes[-1].set_xticklabels(
    [roi_labels[roi] for roi in roi_order],
    rotation=18,
    ha="right",
)
axes[-1].set_xlabel("Область ROI", labelpad=8)

for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)

# ---------------------------------------------------------------------
# Общая легенда
# ---------------------------------------------------------------------
legend_handles = [
    mpatches.Patch(
        facecolor=COL_TBI,
        edgecolor="black",
        label="ЧМТ",
    ),
    mpatches.Patch(
        facecolor=COL_CONTROL,
        edgecolor="black",
        label="Контроль",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, -0.025),
)

# ---------------------------------------------------------------------
# Сохранение
# ---------------------------------------------------------------------
figure_path = OUT["figures"] / "slow_fast_indices_boxplots_by_roi_robust.png"

save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 5.5.1. Проверка экстремальных значений индексов замедления

index_columns = [
    "slow_alpha_ratio",
    "slow_beta_ratio",
    "log_slow_fast_ratio",
]

extreme_rows = []

for col in index_columns:
    tmp = plot_df[
        ["group", "group_norm", "subject_id", "record_id", "region", col]
    ].copy()

    tmp = tmp.replace([np.inf, -np.inf], np.nan).dropna(subset=[col])
    tmp["metric"] = col
    tmp = tmp.sort_values(col, ascending=False).head(15)

    extreme_rows.append(tmp)

extreme_values = pd.concat(extreme_rows, ignore_index=True)

display(extreme_values)

extreme_path = OUT["tables"] / "extreme_slow_fast_indices_top15.csv"
extreme_path.parent.mkdir(parents=True, exist_ok=True)

extreme_values.to_csv(extreme_path, index=False)

print(f"Таблица сохранена: {extreme_path}")

# 6. Интегральные характеристики формы спектра

Помимо мощности отдельных диапазонов рассчитываются характеристики общей формы спектра:

- `SEF50` — частота, ниже которой расположено 50% суммарной мощности;
- `SEF95` — частота, ниже которой расположено 95% суммарной мощности;
- нормированная спектральная энтропия;
- spectral flatness.

Эти признаки помогают описать глобальное смещение спектра и степень его концентрации в отдельных частотных областях.

In [ ]:
# @title 6.1. Функции для SEF, entropy и flatness

def compute_spectral_edge_frequency(
    freqs: np.ndarray,
    psd_values: np.ndarray,
    edge: float,
) -> float:
    """
    Рассчитывает spectral edge frequency.

    Parameters
    ----------
    freqs : np.ndarray
        Частоты.
    psd_values : np.ndarray
        PSD-значения.
    edge : float
        Доля суммарной мощности, например 0.95.

    Returns
    -------
    float
        Частота spectral edge.
    """
    valid_mask = np.isfinite(psd_values) & (psd_values >= 0)

    freqs = freqs[valid_mask]
    psd_values = psd_values[valid_mask]

    if len(freqs) < 2:
        return np.nan

    cumulative_power = np.cumsum(psd_values)
    total_power = cumulative_power[-1]

    if total_power <= 0:
        return np.nan

    cumulative_fraction = cumulative_power / total_power
    index = np.searchsorted(cumulative_fraction, edge)

    index = min(index, len(freqs) - 1)

    return float(freqs[index])


def compute_spectral_entropy(psd_values: np.ndarray) -> float:
    """
    Рассчитывает нормированную спектральную энтропию.

    Parameters
    ----------
    psd_values : np.ndarray
        PSD-значения.

    Returns
    -------
    float
        Нормированная энтропия.
    """
    psd_values = np.asarray(psd_values, dtype=float)
    psd_values = psd_values[np.isfinite(psd_values)]
    psd_values = np.maximum(psd_values, 0)

    total = np.sum(psd_values)

    if total <= 0 or len(psd_values) <= 1:
        return np.nan

    probabilities = psd_values / total
    probabilities = probabilities[probabilities > 0]

    entropy = -np.sum(probabilities * np.log(probabilities))
    normalized_entropy = entropy / np.log(len(psd_values))

    return float(normalized_entropy)


def compute_spectral_flatness(psd_values: np.ndarray) -> float:
    """
    Рассчитывает spectral flatness.

    Parameters
    ----------
    psd_values : np.ndarray
        PSD-значения.

    Returns
    -------
    float
        Spectral flatness.
    """
    psd_values = np.asarray(psd_values, dtype=float)
    psd_values = psd_values[np.isfinite(psd_values)]
    psd_values = np.maximum(psd_values, 1e-20)

    geometric_mean = np.exp(np.mean(np.log(psd_values)))
    arithmetic_mean = np.mean(psd_values)

    if arithmetic_mean <= 0:
        return np.nan

    return float(geometric_mean / arithmetic_mean)

In [ ]:
# @title 6.2. Расчёт характеристик формы спектра

def compute_spectral_shape_features(psd_long: pd.DataFrame) -> pd.DataFrame:
    """
    Рассчитывает SEF, entropy и flatness.

    Parameters
    ----------
    psd_long : pd.DataFrame
        Long-таблица PSD.

    Returns
    -------
    pd.DataFrame
        Таблица характеристик формы спектра.
    """
    records = []

    group_columns = [
        "group",
        "subject_id",
        "record_id",
        "level",
        "region",
    ]

    for keys, sub_df in psd_long.groupby(group_columns):
        freqs = sub_df["freq_hz"].to_numpy()
        psd_values = sub_df["psd"].to_numpy()

        record = dict(zip(group_columns, keys))
        record["sef50"] = compute_spectral_edge_frequency(
            freqs,
            psd_values,
            edge=0.50,
        )
        record["sef95"] = compute_spectral_edge_frequency(
            freqs,
            psd_values,
            edge=0.95,
        )
        record["spectral_entropy"] = compute_spectral_entropy(psd_values)
        record["spectral_flatness"] = compute_spectral_flatness(psd_values)

        records.append(record)

    return pd.DataFrame(records)



In [ ]:
# @title 6.3. Визуализация характеристик формы спектра по ROI

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---------------------------------------------------------------------
# Порядок отображения и подписи
# ---------------------------------------------------------------------
roi_order = ["frontal", "central", "temporal", "parietal", "occipital"]

roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

features_to_plot = [
    {
        "column": "sef50",
        "title": "SEF50",
        "ylabel": "SEF50, Гц",
    },
    {
        "column": "sef95",
        "title": "SEF95",
        "ylabel": "SEF95, Гц",
    },
    {
        "column": "spectral_entropy",
        "title": "Нормированная спектральная энтропия",
        "ylabel": "Энтропия",
    },
    {
        "column": "spectral_flatness",
        "title": "Спектральная плоскостность",
        "ylabel": "Спектральная плоскостность",
    },
]

# ---------------------------------------------------------------------
# Подготовка данных
# ---------------------------------------------------------------------
plot_df = spectral_shape_roi.copy()

plot_df["group_norm"] = plot_df["group"].map(
    {
        "TBI": "tbi",
        "tbi": "tbi",
        "ЧМТ": "tbi",
        "Control": "control",
        "control": "control",
        "Контроль": "control",
    }
)

plot_df = plot_df[plot_df["group_norm"].isin(["tbi", "control"])].copy()

# Проверка наличия нужных колонок.
missing_columns = [
    item["column"]
    for item in features_to_plot
    if item["column"] not in plot_df.columns
]

if missing_columns:
    raise ValueError(
        "В spectral_shape_roi отсутствуют нужные колонки: "
        f"{missing_columns}"
    )

# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
fig, axes = plt.subplots(
    2,
    2,
    figsize=(14, 8.5),
    sharex=False,
    sharey=False,
)

axes = axes.ravel()

fig.suptitle(
    "Характеристики формы спектра по ROI: сравнение групп",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

roi_positions = np.arange(len(roi_order))
offset = 0.18
width = 0.32

for ax, feature in zip(axes, features_to_plot):
    feature_col = feature["column"]

    tbi_data = [
        plot_df.loc[
            (plot_df["group_norm"] == "tbi")
            & (plot_df["region"] == roi_name),
            feature_col,
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        for roi_name in roi_order
    ]

    control_data = [
        plot_df.loc[
            (plot_df["group_norm"] == "control")
            & (plot_df["region"] == roi_name),
            feature_col,
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        for roi_name in roi_order
    ]

    ax.boxplot(
        tbi_data,
        positions=roi_positions - offset,
        widths=width,
        patch_artist=True,
        showfliers=True,
        medianprops={
            "color": "black",
            "linewidth": 1.4,
        },
        boxprops={
            "facecolor": COL_TBI,
            "edgecolor": "black",
            "linewidth": 1.1,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.0,
        },
        capprops={
            "color": "black",
            "linewidth": 1.0,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": COL_TBI,
            "markeredgecolor": COL_TBI,
            "markersize": 2.4,
            "alpha": 0.55,
        },
    )

    ax.boxplot(
        control_data,
        positions=roi_positions + offset,
        widths=width,
        patch_artist=True,
        showfliers=True,
        medianprops={
            "color": "black",
            "linewidth": 1.4,
        },
        boxprops={
            "facecolor": COL_CONTROL,
            "edgecolor": "black",
            "linewidth": 1.1,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.0,
        },
        capprops={
            "color": "black",
            "linewidth": 1.0,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": COL_CONTROL,
            "markeredgecolor": COL_CONTROL,
            "markersize": 2.4,
            "alpha": 0.55,
        },
    )

    ax.set_title(
        feature["title"],
        fontsize=12,
        fontweight="bold",
        pad=6,
    )

    ax.set_ylabel(feature["ylabel"], fontsize=10.5)

    ax.set_xticks(roi_positions)
    ax.set_xticklabels(
        [roi_labels[roi] for roi in roi_order],
        rotation=22,
        ha="right",
    )

    ax.grid(axis="y", linestyle="--", alpha=0.22)
    ax.grid(axis="x", linestyle=":", alpha=0.10)

    # -------------------------------------------------------------
    # Robust-масштаб по Y: данные не меняются, только ось.
    # -------------------------------------------------------------
    values = (
        plot_df[feature_col]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(values) > 0:
        y_low = np.nanpercentile(values, 0.5)
        y_high = np.nanpercentile(values, 99.2)

        y_range = y_high - y_low

        if np.isfinite(y_range) and y_range > 0:
            ax.set_ylim(
                y_low - 0.05 * y_range,
                y_high + 0.10 * y_range,
            )

# ---------------------------------------------------------------------
# Общая легенда снизу
# ---------------------------------------------------------------------
legend_handles = [
    mpatches.Patch(
        facecolor=COL_TBI,
        edgecolor="black",
        label="ЧМТ",
    ),
    mpatches.Patch(
        facecolor=COL_CONTROL,
        edgecolor="black",
        label="Контроль",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 0.015),
)

fig.subplots_adjust(
    left=0.06,
    right=0.98,
    top=0.90,
    bottom=0.13,
    wspace=0.18,
    hspace=0.34,
)

# ---------------------------------------------------------------------
# Сохранение
# ---------------------------------------------------------------------
figure_path = OUT["figures"] / "spectral_shape_features_by_roi.png"

save_figure(fig, figure_path)

plt.show()

# 7. Alpha peak и апериодическая 1/f-компонента

В этом разделе спектр разделяется на два уровня:

1. апериодический фон, описываемый линейной моделью в log-log пространстве;
2. осцилляторные отклонения от этого фона.

Для каждой записи и ROI рассчитываются:

- `aperiodic_exponent`;
- `aperiodic_offset`;
- `alpha_peak_frequency`;
- `alpha_peak_amplitude`;
- `alpha_peak_residual_amplitude`.

In [ ]:
# @title 7.1. Функции alpha peak и апериодической модели

def fit_aperiodic_loglog(
    freqs: np.ndarray,
    psd_values: np.ndarray,
    fit_range: tuple[float, float],
) -> dict[str, np.ndarray | float]:
    """
    Аппроксимирует log10(PSD) линейной моделью от log10(freq).

    Parameters
    ----------
    freqs : np.ndarray
        Частоты.
    psd_values : np.ndarray
        PSD-значения.
    fit_range : tuple[float, float]
        Диапазон частот для фитирования.

    Returns
    -------
    dict[str, np.ndarray | float]
        Параметры апериодического фона и residual spectrum.
    """
    fmin, fmax = fit_range

    mask = (
        (freqs >= fmin)
        & (freqs <= fmax)
        & np.isfinite(psd_values)
        & (psd_values > 0)
    )

    if np.sum(mask) < 3:
        return {
            "aperiodic_exponent": np.nan,
            "aperiodic_offset": np.nan,
            "aperiodic_fit": np.full_like(psd_values, np.nan),
            "residual": np.full_like(psd_values, np.nan),
        }

    log_freqs = np.log10(freqs[mask])
    log_psd = np.log10(psd_values[mask])

    slope, intercept = np.polyfit(log_freqs, log_psd, deg=1)

    full_log_fit = intercept + slope * np.log10(freqs)
    full_fit = 10 ** full_log_fit

    residual = np.log10(psd_values) - full_log_fit

    return {
        "aperiodic_exponent": float(-slope),
        "aperiodic_offset": float(intercept),
        "aperiodic_fit": full_fit,
        "residual": residual,
    }


def compute_alpha_peak(
    freqs: np.ndarray,
    psd_values: np.ndarray,
    residual: np.ndarray | None = None,
    alpha_band: tuple[float, float] = (8.0, 13.0),
) -> dict[str, float]:
    """
    Рассчитывает alpha peak frequency и амплитуду.

    Parameters
    ----------
    freqs : np.ndarray
        Частоты.
    psd_values : np.ndarray
        PSD-значения.
    residual : np.ndarray | None
        Residual spectrum после вычитания апериодики.
    alpha_band : tuple[float, float]
        Альфа-диапазон.

    Returns
    -------
    dict[str, float]
        Метрики alpha peak.
    """
    alpha_min, alpha_max = alpha_band
    mask = (freqs >= alpha_min) & (freqs <= alpha_max)

    if np.sum(mask) == 0:
        return {
            "alpha_peak_frequency": np.nan,
            "alpha_peak_amplitude": np.nan,
            "alpha_peak_residual_amplitude": np.nan,
        }

    alpha_freqs = freqs[mask]
    alpha_psd = psd_values[mask]

    if not np.any(np.isfinite(alpha_psd)):
        return {
            "alpha_peak_frequency": np.nan,
            "alpha_peak_amplitude": np.nan,
            "alpha_peak_residual_amplitude": np.nan,
        }

    peak_index = int(np.nanargmax(alpha_psd))
    peak_freq = alpha_freqs[peak_index]
    peak_amp = alpha_psd[peak_index]

    if residual is not None:
        alpha_residual = residual[mask]
        residual_amp = alpha_residual[peak_index]
    else:
        residual_amp = np.nan

    return {
        "alpha_peak_frequency": float(peak_freq),
        "alpha_peak_amplitude": float(peak_amp),
        "alpha_peak_residual_amplitude": float(residual_amp),
    }

In [ ]:
# @title 7.2. Расчёт alpha peak, 1/f и residual spectrum

def compute_aperiodic_alpha_features(
    psd_long: pd.DataFrame,
    config: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Рассчитывает 1/f параметры, alpha peak и residual spectrum.

    Parameters
    ----------
    psd_long : pd.DataFrame
        Long-таблица PSD.
    config : dict
        Конфигурация анализа.

    Returns
    -------
    tuple[pd.DataFrame, pd.DataFrame]
        Таблица признаков и long-таблица residual spectrum.
    """
    feature_records = []
    residual_records = []

    group_columns = [
        "group",
        "subject_id",
        "record_id",
        "level",
        "region",
    ]

    for keys, sub_df in psd_long.groupby(group_columns):
        freqs = sub_df["freq_hz"].to_numpy()
        psd_values = sub_df["psd"].to_numpy()

        fit_result = fit_aperiodic_loglog(
            freqs=freqs,
            psd_values=psd_values,
            fit_range=config["aperiodic_fit_range"],
        )

        alpha_result = compute_alpha_peak(
            freqs=freqs,
            psd_values=psd_values,
            residual=fit_result["residual"],
        )

        record = dict(zip(group_columns, keys))
        record["aperiodic_exponent"] = fit_result["aperiodic_exponent"]
        record["aperiodic_offset"] = fit_result["aperiodic_offset"]
        record.update(alpha_result)

        feature_records.append(record)

        for freq, residual_value in zip(freqs, fit_result["residual"]):
            residual_records.append(
                {
                    **record,
                    "freq_hz": float(freq),
                    "residual_log_power": float(residual_value),
                }
            )

    return pd.DataFrame(feature_records), pd.DataFrame(residual_records)


aperiodic_alpha_global, residual_global_long = (
    compute_aperiodic_alpha_features(psd_global_long, CONFIG)
)

aperiodic_alpha_roi, residual_roi_long = (
    compute_aperiodic_alpha_features(psd_roi_long, CONFIG)
)

save_table(
    aperiodic_alpha_global,
    OUT["tables"] / "aperiodic_alpha_global.csv",
)

save_table(
    aperiodic_alpha_roi,
    OUT["tables"] / "aperiodic_alpha_roi.csv",
)

save_table(
    residual_roi_long,
    OUT["tables"] / "residual_roi_long.csv",
)

display(aperiodic_alpha_roi.head())

In [ ]:
# @title 7.3. Визуализация 1/f параметров по ROI

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---------------------------------------------------------------------
# Порядок отображения и подписи
# ---------------------------------------------------------------------
roi_order = ["frontal", "central", "temporal", "parietal", "occipital"]

roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

features_to_plot = [
    {
        "column": "aperiodic_exponent",
        "title": "Апериодический показатель 1/f ",
        "ylabel": "Апериодический показатель (exponent)",
    },
    {
        "column": "aperiodic_offset",
        "title": "Апериодический offset",
        "ylabel": "Апериодический показатель (offset)",
    },
]

# ---------------------------------------------------------------------
# Подготовка данных
# ---------------------------------------------------------------------
plot_df = aperiodic_alpha_roi.copy()

plot_df["group_norm"] = plot_df["group"].map(
    {
        "TBI": "tbi",
        "tbi": "tbi",
        "ЧМТ": "tbi",
        "Control": "control",
        "control": "control",
        "Контроль": "control",
    }
)

plot_df = plot_df[plot_df["group_norm"].isin(["tbi", "control"])].copy()

missing_columns = [
    item["column"]
    for item in features_to_plot
    if item["column"] not in plot_df.columns
]

if missing_columns:
    raise ValueError(
        "В aperiodic_alpha_roi отсутствуют нужные колонки: "
        f"{missing_columns}"
    )

# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5.2),
    sharex=False,
    sharey=False,
)

fig.suptitle(
    "Апериодические 1/f параметры по ROI: сравнение групп",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

roi_positions = np.arange(len(roi_order))
offset = 0.18
width = 0.32

for ax, feature in zip(axes, features_to_plot):
    feature_col = feature["column"]

    tbi_data = [
        plot_df.loc[
            (plot_df["group_norm"] == "tbi")
            & (plot_df["region"] == roi_name),
            feature_col,
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        for roi_name in roi_order
    ]

    control_data = [
        plot_df.loc[
            (plot_df["group_norm"] == "control")
            & (plot_df["region"] == roi_name),
            feature_col,
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        for roi_name in roi_order
    ]

    ax.boxplot(
        tbi_data,
        positions=roi_positions - offset,
        widths=width,
        patch_artist=True,
        showfliers=True,
        medianprops={
            "color": "black",
            "linewidth": 1.4,
        },
        boxprops={
            "facecolor": COL_TBI,
            "edgecolor": "black",
            "linewidth": 1.1,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.0,
        },
        capprops={
            "color": "black",
            "linewidth": 1.0,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": COL_TBI,
            "markeredgecolor": COL_TBI,
            "markersize": 2.4,
            "alpha": 0.55,
        },
    )

    ax.boxplot(
        control_data,
        positions=roi_positions + offset,
        widths=width,
        patch_artist=True,
        showfliers=True,
        medianprops={
            "color": "black",
            "linewidth": 1.4,
        },
        boxprops={
            "facecolor": COL_CONTROL,
            "edgecolor": "black",
            "linewidth": 1.1,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.0,
        },
        capprops={
            "color": "black",
            "linewidth": 1.0,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": COL_CONTROL,
            "markeredgecolor": COL_CONTROL,
            "markersize": 2.4,
            "alpha": 0.55,
        },
    )

    ax.set_title(
        feature["title"],
        fontsize=12,
        fontweight="bold",
        pad=6,
    )

    ax.set_ylabel(feature["ylabel"], fontsize=10.5)

    ax.set_xticks(roi_positions)
    ax.set_xticklabels(
        [roi_labels[roi] for roi in roi_order],
        rotation=22,
        ha="right",
    )

    ax.grid(axis="y", linestyle="--", alpha=0.22)
    ax.grid(axis="x", linestyle=":", alpha=0.10)

    values = (
        plot_df[feature_col]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(values) > 0:
        y_low = np.nanpercentile(values, 0.5)
        y_high = np.nanpercentile(values, 99.2)
        y_range = y_high - y_low

        if np.isfinite(y_range) and y_range > 0:
            ax.set_ylim(
                y_low - 0.05 * y_range,
                y_high + 0.10 * y_range,
            )

# ---------------------------------------------------------------------
# Общая легенда
# ---------------------------------------------------------------------
legend_handles = [
    mpatches.Patch(
        facecolor=COL_TBI,
        edgecolor="black",
        label="ЧМТ",
    ),
    mpatches.Patch(
        facecolor=COL_CONTROL,
        edgecolor="black",
        label="Контроль",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 0.015),
)

fig.subplots_adjust(
    left=0.06,
    right=0.98,
    top=0.86,
    bottom=0.20,
    wspace=0.18,
)

# ---------------------------------------------------------------------
# Сохранение
# ---------------------------------------------------------------------
figure_path = OUT["figures"] / "aperiodic_1f_features_by_roi.png"

save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 7.4. Визуализация alpha peak признаков по ROI

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---------------------------------------------------------------------
# Порядок отображения и подписи
# ---------------------------------------------------------------------
roi_order = ["frontal", "central", "temporal", "parietal", "occipital"]

roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

# ---------------------------------------------------------------------
# Подготовка данных
# ---------------------------------------------------------------------
plot_df = aperiodic_alpha_roi.copy()

plot_df["group_norm"] = plot_df["group"].map(
    {
        "TBI": "tbi",
        "tbi": "tbi",
        "ЧМТ": "tbi",
        "Control": "control",
        "control": "control",
        "Контроль": "control",
    }
)

plot_df = plot_df[plot_df["group_norm"].isin(["tbi", "control"])].copy()

# ---------------------------------------------------------------------
# Автоматический поиск alpha peak колонок
# ---------------------------------------------------------------------
candidate_features = [
    {
        "candidates": [
            "alpha_peak_freq",
            "alpha_peak_frequency",
            "paf",
            "peak_alpha_frequency",
        ],
        "title": "Частота α-пика",
        "ylabel": "Частота, Гц",
        "transform": None,
    },
    {
        "candidates": [
            "alpha_peak_power",
            "alpha_peak_amplitude",
            "alpha_peak_height",
            "alpha_peak_value",
        ],
        "title": "Амплитуда α-пика",
        "ylabel": "log10(амплитуда α-пика)",
        "transform": "log10",
    },
    {
        "candidates": [
            "alpha_peak_residual",
            "alpha_residual_peak",
            "alpha_peak_over_aperiodic",
            "alpha_peak_prominence",
        ],
        "title": "α-пик над 1/f компонентой",
        "ylabel": "Residual peak",
        "transform": None,
    },
    {
        "candidates": [
            "alpha_peak_width",
            "alpha_width",
            "alpha_peak_bandwidth",
        ],
        "title": "Ширина α-пика",
        "ylabel": "Ширина, Гц",
        "transform": None,
    },
]

features_to_plot = []

for feature in candidate_features:
    found_col = None

    for col in feature["candidates"]:
        if col in plot_df.columns:
            found_col = col
            break

    if found_col is not None:
        features_to_plot.append(
            {
                "column": found_col,
                "title": feature["title"],
                "ylabel": feature["ylabel"],
                "transform": feature["transform"],
            }
        )

if not features_to_plot:
    raise ValueError(
        "Не найдены alpha peak колонки в aperiodic_alpha_roi.\n"
        "Проверь названия колонок:\n"
        f"{list(plot_df.columns)}"
    )

# ---------------------------------------------------------------------
# Создаём transformed-колонки для визуализации
# ---------------------------------------------------------------------
for feature in features_to_plot:
    source_col = feature["column"]
    plot_col = source_col

    if feature["transform"] == "log10":
        plot_col = f"log10_{source_col}"

        values = (
            plot_df[source_col]
            .replace([np.inf, -np.inf], np.nan)
        )

        positive_values = values.where(values > 0)

        plot_df[plot_col] = np.log10(positive_values)

    feature["plot_column"] = plot_col

print("Будут построены alpha peak признаки:")
for item in features_to_plot:
    print(
        f"- {item['title']}: колонка `{item['column']}`"
        + (
            " → log10 для визуализации"
            if item["transform"] == "log10"
            else ""
        )
    )

# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
n_features = len(features_to_plot)
n_cols = 2
n_rows = int(np.ceil(n_features / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(14, 4.8 * n_rows),
    sharex=False,
    sharey=False,
)

axes = np.atleast_1d(axes).ravel()

fig.suptitle(
    "Характеристики α-пика по ROI: сравнение групп",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

roi_positions = np.arange(len(roi_order))
offset = 0.18
width = 0.32

for idx, feature in enumerate(features_to_plot):
    ax = axes[idx]
    feature_col = feature["plot_column"]

    tbi_data = [
        plot_df.loc[
            (plot_df["group_norm"] == "tbi")
            & (plot_df["region"] == roi_name),
            feature_col,
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        for roi_name in roi_order
    ]

    control_data = [
        plot_df.loc[
            (plot_df["group_norm"] == "control")
            & (plot_df["region"] == roi_name),
            feature_col,
        ]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        for roi_name in roi_order
    ]

    ax.boxplot(
        tbi_data,
        positions=roi_positions - offset,
        widths=width,
        patch_artist=True,
        showfliers=True,
        medianprops={
            "color": "black",
            "linewidth": 1.4,
        },
        boxprops={
            "facecolor": COL_TBI,
            "edgecolor": "black",
            "linewidth": 1.1,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.0,
        },
        capprops={
            "color": "black",
            "linewidth": 1.0,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": COL_TBI,
            "markeredgecolor": COL_TBI,
            "markersize": 2.4,
            "alpha": 0.55,
        },
    )

    ax.boxplot(
        control_data,
        positions=roi_positions + offset,
        widths=width,
        patch_artist=True,
        showfliers=True,
        medianprops={
            "color": "black",
            "linewidth": 1.4,
        },
        boxprops={
            "facecolor": COL_CONTROL,
            "edgecolor": "black",
            "linewidth": 1.1,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.0,
        },
        capprops={
            "color": "black",
            "linewidth": 1.0,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": COL_CONTROL,
            "markeredgecolor": COL_CONTROL,
            "markersize": 2.4,
            "alpha": 0.55,
        },
    )

    ax.set_title(
        feature["title"],
        fontsize=12,
        fontweight="bold",
        pad=6,
    )

    ax.set_ylabel(feature["ylabel"], fontsize=10.5)

    ax.set_xticks(roi_positions)
    ax.set_xticklabels(
        [roi_labels[roi] for roi in roi_order],
        rotation=20,
        ha="right",
    )

    ax.grid(axis="y", linestyle="--", alpha=0.22)
    ax.grid(axis="x", linestyle=":", alpha=0.10)

    # -------------------------------------------------------------
    # Robust-масштаб по Y: данные не меняются, только ось.
    # -------------------------------------------------------------
    values = (
        plot_df[feature_col]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if len(values) > 0:
        y_low = np.nanpercentile(values, 0.5)
        y_high = np.nanpercentile(values, 99.2)

        y_range = y_high - y_low

        if np.isfinite(y_range) and y_range > 0:
            ax.set_ylim(
                y_low - 0.05 * y_range,
                y_high + 0.10 * y_range,
            )

# Выключаем пустые панели.
for ax in axes[n_features:]:
    ax.axis("off")

# ---------------------------------------------------------------------
# Общая легенда
# ---------------------------------------------------------------------
legend_handles = [
    mpatches.Patch(
        facecolor=COL_TBI,
        edgecolor="black",
        label="ЧМТ",
    ),
    mpatches.Patch(
        facecolor=COL_CONTROL,
        edgecolor="black",
        label="Контроль",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, 0.01),
)

fig.subplots_adjust(
    left=0.06,
    right=0.98,
    top=0.87,
    bottom=0.18,
    wspace=0.18,
    hspace=0.36,
)

# ---------------------------------------------------------------------
# Сохранение
# ---------------------------------------------------------------------
figure_path = OUT["figures"] / "alpha_peak_features_by_roi.png"

save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 7.5. Residual spectrum после удаления 1/f компоненты по ROI

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ---------------------------------------------------------------------
# Порядок отображения и подписи
# ---------------------------------------------------------------------
roi_order = ["frontal", "central", "temporal", "parietal", "occipital"]

roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

# ---------------------------------------------------------------------
# Подготовка данных
# ---------------------------------------------------------------------
plot_df = residual_roi_long.copy()

plot_df["group_norm"] = plot_df["group"].map(
    {
        "TBI": "tbi",
        "tbi": "tbi",
        "ЧМТ": "tbi",
        "Control": "control",
        "control": "control",
        "Контроль": "control",
    }
)

plot_df = plot_df[plot_df["group_norm"].isin(["tbi", "control"])].copy()

required_columns = [
    "group_norm",
    "region",
    "freq_hz",
    "residual_log_power",
]

missing_columns = [
    col for col in required_columns
    if col not in plot_df.columns
]

if missing_columns:
    raise ValueError(
        "В residual_roi_long отсутствуют нужные колонки: "
        f"{missing_columns}"
    )

# ---------------------------------------------------------------------
# Агрегация: среднее и SEM по record/subject-level строкам
# ---------------------------------------------------------------------
summary = (
    plot_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["residual_log_power"])
    .groupby(["group_norm", "region", "freq_hz"])
    .agg(
        mean_residual=("residual_log_power", "mean"),
        std_residual=("residual_log_power", "std"),
        n=("residual_log_power", "count"),
    )
    .reset_index()
)

summary["sem_residual"] = summary["std_residual"] / np.sqrt(summary["n"])

# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
fig, axes = plt.subplots(
    2,
    3,
    figsize=(15.5, 8.2),
    sharex=True,
    sharey=True,
)

axes = axes.ravel()

fig.suptitle(
    "Residual spectrum после удаления 1/f компоненты по ROI",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

for idx, roi_name in enumerate(roi_order):
    ax = axes[idx]

    for group_norm, color, label in [
        ("tbi", COL_TBI, "ЧМТ"),
        ("control", COL_CONTROL, "Контроль"),
    ]:
        sub = summary[
            (summary["region"] == roi_name)
            & (summary["group_norm"] == group_norm)
        ].sort_values("freq_hz")

        if sub.empty:
            continue

        x = sub["freq_hz"].to_numpy()
        y = sub["mean_residual"].to_numpy()
        sem = sub["sem_residual"].to_numpy()

        ax.plot(
            x,
            y,
            color=color,
            linewidth=1.8,
            label=label,
        )

        ax.fill_between(
            x,
            y - sem,
            y + sem,
            color=color,
            alpha=0.16,
            linewidth=0,
        )

    ax.axhline(
        0,
        color="black",
        linewidth=0.8,
        linestyle="--",
        alpha=0.6,
    )

    ax.axvspan(
        8,
        13,
        color="gray",
        alpha=0.10,
        linewidth=0,
    )

    ax.set_title(
        roi_labels[roi_name],
        fontsize=12,
        fontweight="bold",
        pad=6,
    )

    ax.grid(axis="y", linestyle="--", alpha=0.22)
    ax.grid(axis="x", linestyle=":", alpha=0.10)

    ax.set_xlabel("Частота, Гц")
    ax.set_ylabel("Остаточная log-мощность")

# Пустую шестую панель выключаем.
axes[-1].axis("off")

# ---------------------------------------------------------------------
# Общая легенда снизу
# ---------------------------------------------------------------------
legend_handles = [
    mpatches.Patch(
        facecolor=COL_TBI,
        edgecolor="black",
        label="ЧМТ",
    ),
    mpatches.Patch(
        facecolor=COL_CONTROL,
        edgecolor="black",
        label="Контроль",
    ),
    mpatches.Patch(
        facecolor="gray",
        edgecolor="none",
        alpha=0.20,
        label="α-диапазон, 8–13 Гц",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=3,
    frameon=False,
    bbox_to_anchor=(0.5, 0.015),
)

fig.subplots_adjust(
    left=0.06,
    right=0.98,
    top=0.90,
    bottom=0.13,
    wspace=0.18,
    hspace=0.34,
)

# ---------------------------------------------------------------------
# Сохранение
# ---------------------------------------------------------------------
figure_path = OUT["figures"] / "residual_spectrum_by_roi.png"

save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 7.6. Heatmap различий residual spectrum: ЧМТ − Контроль

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ---------------------------------------------------------------------
# Порядок отображения и подписи
# ---------------------------------------------------------------------
roi_order = ["frontal", "central", "temporal", "parietal", "occipital"]

roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

# ---------------------------------------------------------------------
# Подготовка данных
# ---------------------------------------------------------------------
plot_df = residual_roi_long.copy()

plot_df["group_norm"] = plot_df["group"].map(
    {
        "TBI": "tbi",
        "tbi": "tbi",
        "ЧМТ": "tbi",
        "Control": "control",
        "control": "control",
        "Контроль": "control",
    }
)

plot_df = plot_df[plot_df["group_norm"].isin(["tbi", "control"])].copy()

mean_residual = (
    plot_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["residual_log_power"])
    .groupby(["group_norm", "region", "freq_hz"])["residual_log_power"]
    .mean()
    .reset_index()
)

tbi = mean_residual[mean_residual["group_norm"] == "tbi"]
control = mean_residual[mean_residual["group_norm"] == "control"]

merged = tbi.merge(
    control,
    on=["region", "freq_hz"],
    suffixes=("_tbi", "_control"),
)

merged["diff_tbi_minus_control"] = (
    merged["residual_log_power_tbi"]
    - merged["residual_log_power_control"]
)

pivot = (
    merged
    .pivot(index="region", columns="freq_hz", values="diff_tbi_minus_control")
    .loc[roi_order]
)

freqs = pivot.columns.to_numpy(dtype=float)

diff_abs = np.nanmax(np.abs(pivot.values))
norm = TwoSlopeNorm(vmin=-diff_abs, vcenter=0, vmax=diff_abs)

# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 4.8))

im = ax.imshow(
    pivot.values,
    aspect="auto",
    cmap="RdBu_r",
    norm=norm,
    extent=[
        freqs.min(),
        freqs.max(),
        len(roi_order) - 0.5,
        -0.5,
    ],
)

ax.set_title(
    "Разность residual spectrum после удаления 1/f компоненты: ЧМТ − Контроль",
    fontsize=14,
    fontweight="bold",
    pad=10,
)

ax.set_xlabel("Частота, Гц")
ax.set_ylabel("ROI")

ax.set_yticks(np.arange(len(roi_order)))
ax.set_yticklabels([roi_labels[roi] for roi in roi_order])

ax.axvspan(
    8,
    13,
    color="gray",
    alpha=0.16,
    linewidth=0,
)

ax.text(
    10.5,
    -0.75,
    "α",
    ha="center",
    va="center",
    fontsize=11,
    fontweight="bold",
    color="dimgray",
)

cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label("Разность residual log power")

fig.subplots_adjust(
    left=0.08,
    right=0.95,
    top=0.84,
    bottom=0.18,
)

# ---------------------------------------------------------------------
# Сохранение
# ---------------------------------------------------------------------
figure_path = OUT["figures"] / "residual_spectrum_difference_heatmap_tbi_minus_control.png"

save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 7.7. Частотно-кластерный анализ residual spectrum на глобальном уровне

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


# ---------------------------------------------------------------------
# Настройки анализа
# ---------------------------------------------------------------------
N_PERMUTATIONS = 5000
ALPHA_CLUSTER = 0.05
ALPHA_POINT = 0.05
RANDOM_STATE = CONFIG.get("random_state", 97)

rng = np.random.default_rng(RANDOM_STATE)


# ---------------------------------------------------------------------
# Подготовка данных
# ---------------------------------------------------------------------
cluster_df = residual_global_long.copy()

cluster_df["group_norm"] = cluster_df["group"].map(
    {
        "TBI": "tbi",
        "tbi": "tbi",
        "ЧМТ": "tbi",
        "Control": "control",
        "control": "control",
        "Контроль": "control",
    }
)

cluster_df = cluster_df[cluster_df["group_norm"].isin(["tbi", "control"])].copy()

required_columns = [
    "group_norm",
    "subject_id",
    "freq_hz",
    "residual_log_power",
]

missing_columns = [
    col for col in required_columns
    if col not in cluster_df.columns
]

if missing_columns:
    raise ValueError(
        "В residual_global_long отсутствуют нужные колонки: "
        f"{missing_columns}"
    )

cluster_df = (
    cluster_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["residual_log_power", "freq_hz"])
    .copy()
)

# ---------------------------------------------------------------------
# Агрегация на уровне субъектов
# ---------------------------------------------------------------------
# Если у субъекта несколько записей, сначала усредняем residual по частоте.
# Это важно, чтобы один субъект не давал непропорционально большой вклад.
# ---------------------------------------------------------------------
subject_freq = (
    cluster_df
    .groupby(["group_norm", "subject_id", "freq_hz"], as_index=False)
    .agg(
        residual_log_power=("residual_log_power", "mean"),
    )
)

freqs = np.sort(subject_freq["freq_hz"].unique())

tbi_subjects = (
    subject_freq.loc[subject_freq["group_norm"] == "tbi", "subject_id"]
    .drop_duplicates()
    .sort_values()
    .to_numpy()
)

control_subjects = (
    subject_freq.loc[subject_freq["group_norm"] == "control", "subject_id"]
    .drop_duplicates()
    .sort_values()
    .to_numpy()
)

print(f"Число субъектов ЧМТ: {len(tbi_subjects)}")
print(f"Число субъектов контроля: {len(control_subjects)}")
print(f"Число частотных точек: {len(freqs)}")


def make_subject_matrix(df, subjects, freqs):
    """
    Формирует матрицу subject × frequency.

    Parameters
    ----------
    df : pd.DataFrame
        Таблица residual spectrum.
    subjects : array-like
        Список субъектов.
    freqs : array-like
        Список частот.

    Returns
    -------
    np.ndarray
        Матрица размера n_subjects × n_freqs.
    """
    pivot = (
        df[df["subject_id"].isin(subjects)]
        .pivot_table(
            index="subject_id",
            columns="freq_hz",
            values="residual_log_power",
            aggfunc="mean",
        )
        .reindex(index=subjects, columns=freqs)
    )

    return pivot.to_numpy(dtype=float)


x_tbi = make_subject_matrix(subject_freq, tbi_subjects, freqs)
x_control = make_subject_matrix(subject_freq, control_subjects, freqs)


# ---------------------------------------------------------------------
# Функции для t-статистики и кластеров
# ---------------------------------------------------------------------
def compute_t_by_frequency(x_a, x_b):
    """
    Welch t-test по каждой частоте.

    Parameters
    ----------
    x_a, x_b : np.ndarray
        Матрицы subject × frequency.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        t-статистика и p-value по частотам.
    """
    t_values = np.full(x_a.shape[1], np.nan)
    p_values = np.full(x_a.shape[1], np.nan)

    for idx in range(x_a.shape[1]):
        a = x_a[:, idx]
        b = x_b[:, idx]

        a = a[np.isfinite(a)]
        b = b[np.isfinite(b)]

        if len(a) < 3 or len(b) < 3:
            continue

        t_stat, p_val = stats.ttest_ind(
            a,
            b,
            equal_var=False,
            nan_policy="omit",
        )

        t_values[idx] = t_stat
        p_values[idx] = p_val

    return t_values, p_values


def find_clusters(mask):
    """
    Находит непрерывные кластеры True-значений.

    Parameters
    ----------
    mask : np.ndarray
        Булева маска значимых частот.

    Returns
    -------
    list[tuple[int, int]]
        Список кластеров как интервалы индексов [start, stop).
    """
    clusters = []
    start = None

    for idx, value in enumerate(mask):
        if value and start is None:
            start = idx

        if (not value or idx == len(mask) - 1) and start is not None:
            stop = idx if not value else idx + 1
            clusters.append((start, stop))
            start = None

    return clusters


def cluster_mass(t_values, cluster):
    """
    Считает массу кластера как сумму |t| внутри кластера.
    """
    start, stop = cluster
    return np.nansum(np.abs(t_values[start:stop]))


# ---------------------------------------------------------------------
# Наблюдаемая статистика
# ---------------------------------------------------------------------
t_obs, p_obs = compute_t_by_frequency(x_tbi, x_control)

point_mask = p_obs < ALPHA_POINT
observed_clusters = find_clusters(point_mask)

observed_cluster_masses = np.array(
    [
        cluster_mass(t_obs, cluster)
        for cluster in observed_clusters
    ],
    dtype=float,
)

print(f"Найдено первичных кластеров: {len(observed_clusters)}")


# ---------------------------------------------------------------------
# Permutation cluster correction
# ---------------------------------------------------------------------
x_all = np.vstack([x_tbi, x_control])
labels = np.array(
    ["tbi"] * x_tbi.shape[0]
    + ["control"] * x_control.shape[0]
)

n_tbi = x_tbi.shape[0]
max_cluster_masses = np.zeros(N_PERMUTATIONS)

for perm_idx in range(N_PERMUTATIONS):
    permuted_labels = rng.permutation(labels)

    perm_tbi = x_all[permuted_labels == "tbi"]
    perm_control = x_all[permuted_labels == "control"]

    t_perm, p_perm = compute_t_by_frequency(perm_tbi, perm_control)

    perm_mask = p_perm < ALPHA_POINT
    perm_clusters = find_clusters(perm_mask)

    if perm_clusters:
        masses = [
            cluster_mass(t_perm, cluster)
            for cluster in perm_clusters
        ]
        max_cluster_masses[perm_idx] = np.nanmax(masses)
    else:
        max_cluster_masses[perm_idx] = 0.0


# ---------------------------------------------------------------------
# p-value для кластеров
# ---------------------------------------------------------------------
cluster_results = []

for cluster, mass in zip(observed_clusters, observed_cluster_masses):
    p_cluster = (
        np.sum(max_cluster_masses >= mass) + 1
    ) / (N_PERMUTATIONS + 1)

    start, stop = cluster

    cluster_results.append(
        {
            "cluster_start_idx": start,
            "cluster_stop_idx": stop - 1,
            "freq_start_hz": freqs[start],
            "freq_stop_hz": freqs[stop - 1],
            "cluster_mass": mass,
            "cluster_p": p_cluster,
            "significant": p_cluster < ALPHA_CLUSTER,
        }
    )

cluster_results_df = pd.DataFrame(cluster_results)

display(cluster_results_df)

cluster_results_path = (
    OUT["tables"]
    / "residual_global_frequency_cluster_results.csv"
)

cluster_results_df.to_csv(cluster_results_path, index=False)

print(f"Таблица кластеров сохранена: {cluster_results_path}")


# ---------------------------------------------------------------------
# Средние кривые и SEM для визуализации
# ---------------------------------------------------------------------
def mean_sem(matrix):
    """Возвращает mean и SEM по субъектам."""
    mean = np.nanmean(matrix, axis=0)

    n = np.sum(np.isfinite(matrix), axis=0)
    std = np.nanstd(matrix, axis=0, ddof=1)

    sem = std / np.sqrt(n)

    return mean, sem


mean_tbi, sem_tbi = mean_sem(x_tbi)
mean_control, sem_control = mean_sem(x_control)


# ---------------------------------------------------------------------
# Значимые кластеры
# ---------------------------------------------------------------------
significant_clusters = [
    (int(row["cluster_start_idx"]), int(row["cluster_stop_idx"]) + 1)
    for _, row in cluster_results_df.iterrows()
    if bool(row["significant"])
]


# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
fig, axes = plt.subplots(
    2,
    1,
    figsize=(11.5, 7.2),
    sharex=True,
)

fig.suptitle(
    "Частотно-кластерный анализ residual-спектра после удаления 1/f компоненты",
    fontsize=14,
    fontweight="bold",
    y=0.98,
)

ax_top, ax_bottom = axes


# ---------------------------------------------------------------------
# A. Средний residual spectrum
# ---------------------------------------------------------------------
ax_top.plot(
    freqs,
    mean_tbi,
    color=COL_TBI,
    linewidth=2.0,
    label="ЧМТ",
)

ax_top.fill_between(
    freqs,
    mean_tbi - sem_tbi,
    mean_tbi + sem_tbi,
    color=COL_TBI,
    alpha=0.16,
    linewidth=0,
)

ax_top.plot(
    freqs,
    mean_control,
    color=COL_CONTROL,
    linewidth=2.0,
    label="Контроль",
)

ax_top.fill_between(
    freqs,
    mean_control - sem_control,
    mean_control + sem_control,
    color=COL_CONTROL,
    alpha=0.16,
    linewidth=0,
)

for start, stop in significant_clusters:
    ax_top.axvspan(
        freqs[start],
        freqs[stop - 1],
        color=COL_TBI,
        alpha=0.08,
        linewidth=0,
    )

ax_top.axhline(
    0,
    color="black",
    linewidth=0.8,
    linestyle="--",
    alpha=0.5,
)

ax_top.set_title(
    "A. Средний residual-спектр по группам",
    fontsize=12,
    fontweight="bold",
    loc="left",
)

ax_top.set_ylabel("Residual log power")
ax_top.legend(frameon=False, loc="upper right")

ax_top.grid(axis="y", linestyle="--", alpha=0.22)
ax_top.grid(axis="x", linestyle=":", alpha=0.10)


# ---------------------------------------------------------------------
# B. t-статистика
# ---------------------------------------------------------------------
ax_bottom.plot(
    freqs,
    t_obs,
    color=COL_TBI,
    linewidth=1.8,
    label="t-статистика",
)

ax_bottom.axhline(
    0,
    color="black",
    linewidth=0.8,
    linestyle="--",
    alpha=0.6,
)

for start, stop in significant_clusters:
    ax_bottom.axvspan(
        freqs[start],
        freqs[stop - 1],
        color=COL_TBI,
        alpha=0.08,
        linewidth=0,
    )

ax_bottom.set_title(
    "B. t-статистика по частоте",
    fontsize=12,
    fontweight="bold",
    loc="left",
)

ax_bottom.set_xlabel("Частота, Гц")
ax_bottom.set_ylabel("t-статистика")

ax_bottom.grid(axis="y", linestyle="--", alpha=0.22)
ax_bottom.grid(axis="x", linestyle=":", alpha=0.10)

ax_bottom.legend(frameon=False, loc="upper right")


# ---------------------------------------------------------------------
# Отметка α-диапазона
# ---------------------------------------------------------------------
for ax in axes:
    ax.axvspan(
        8,
        13,
        color="gray",
        alpha=0.08,
        linewidth=0,
    )

    ax.text(
        10.5,
        0.96,
        "α",
        transform=ax.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=10,
        fontweight="bold",
        color="dimgray",
    )


fig.subplots_adjust(
    left=0.08,
    right=0.98,
    top=0.90,
    bottom=0.10,
    hspace=0.32,
)

# ---------------------------------------------------------------------
# Сохранение
# ---------------------------------------------------------------------
figure_path = (
    OUT["figures"]
    / "residual_global_frequency_cluster_analysis.png"
)

save_figure(fig, figure_path)

plt.show()

# 8. Статистическое сравнение групп

На этом этапе для рассчитанных спектральных признаков выполняется межгрупповое сравнение.

Для каждого признака рассчитываются:

- среднее значение в группе ЧМТ;
- среднее значение в контрольной группе;
- разность средних;
- Hedges' g;
- Mann–Whitney p-value;
- FDR-corrected q-value.

Единицей сравнения в этой таблице является запись. Для финальной ML-оценки и train/test-разбиений необходимо использовать subject-level split.

In [ ]:
# @title 8.1. Статистические функции

def hedges_g(x: np.ndarray, y: np.ndarray) -> float:
    """
    Рассчитывает Hedges' g для двух независимых групп.

    Parameters
    ----------
    x : np.ndarray
        Значения первой группы.
    y : np.ndarray
        Значения второй группы.

    Returns
    -------
    float
        Hedges' g.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    n_x = len(x)
    n_y = len(y)

    if n_x < 2 or n_y < 2:
        return np.nan

    pooled_sd = np.sqrt(
        (
            (n_x - 1) * np.var(x, ddof=1)
            + (n_y - 1) * np.var(y, ddof=1)
        )
        / (n_x + n_y - 2)
    )

    if pooled_sd == 0:
        return np.nan

    cohen_d = (np.mean(x) - np.mean(y)) / pooled_sd
    correction = 1 - 3 / (4 * (n_x + n_y) - 9)

    return float(cohen_d * correction)


def rank_biserial_from_u(
    u_stat: float,
    n_x: int,
    n_y: int,
) -> float:
    """
    Рассчитывает rank-biserial correlation из статистики Mann–Whitney U.

    Значение > 0 означает, что значения первой группы в среднем выше,
    чем значения второй группы.

    Parameters
    ----------
    u_stat : float
        Статистика U для первой группы.
    n_x : int
        Размер первой группы.
    n_y : int
        Размер второй группы.

    Returns
    -------
    float
        Rank-biserial correlation.
    """
    if n_x == 0 or n_y == 0:
        return np.nan

    return float((2 * u_stat) / (n_x * n_y) - 1)


def normalize_stat_group(value: str) -> str:
    """
    Приводит названия групп к TBI / Control.
    """
    mapping = {
        "TBI": "TBI",
        "tbi": "TBI",
        "ЧМТ": "TBI",
        "patient": "TBI",
        "patients": "TBI",
        "Control": "Control",
        "control": "Control",
        "Контроль": "Control",
        "CTRL": "Control",
        "ctl": "Control",
        "HC": "Control",
        "healthy": "Control",
    }

    return mapping.get(str(value).strip(), str(value).strip())


def safe_nanmean(values: np.ndarray) -> float:
    """Безопасно считает среднее."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan

    return float(np.nanmean(values))


def safe_nanmedian(values: np.ndarray) -> float:
    """Безопасно считает медиану."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan

    return float(np.nanmedian(values))


def safe_iqr(values: np.ndarray) -> float:
    """Безопасно считает IQR."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan

    return float(
        np.nanpercentile(values, 75)
        - np.nanpercentile(values, 25)
    )


def compare_groups_by_features(
    df: pd.DataFrame,
    feature_columns: list[str],
    context_columns: list[str] | None = None,
) -> pd.DataFrame:
    """
    Сравнивает группы TBI и Control по набору признаков.

    Parameters
    ----------
    df : pd.DataFrame
        Таблица признаков. Должна содержать колонку group.
    feature_columns : list[str]
        Список числовых признаков.
    context_columns : list[str] | None
        Контекстные колонки для группировки, например region.

    Returns
    -------
    pd.DataFrame
        Таблица статистических сравнений.
    """
    context_columns = context_columns or []

    if "group" not in df.columns:
        raise ValueError("В таблице df отсутствует обязательная колонка 'group'.")

    missing_features = [
        feature for feature in feature_columns
        if feature not in df.columns
    ]

    if missing_features:
        raise ValueError(
            "В таблице df отсутствуют признаки: "
            f"{missing_features}"
        )

    work_df = df.copy()
    work_df["group"] = work_df["group"].map(normalize_stat_group)

    records = []

    if context_columns:
        iterator = work_df.groupby(context_columns, dropna=False)
    else:
        iterator = [((), work_df)]

    for context_key, sub_df in iterator:
        if context_columns and not isinstance(context_key, tuple):
            context_key = (context_key,)

        if context_columns:
            context = dict(zip(context_columns, context_key))
        else:
            context = {}

        for feature in feature_columns:
            tbi_values = (
                sub_df.loc[sub_df["group"] == "TBI", feature]
                .to_numpy(dtype=float)
            )
            ctl_values = (
                sub_df.loc[sub_df["group"] == "Control", feature]
                .to_numpy(dtype=float)
            )

            tbi_values = tbi_values[np.isfinite(tbi_values)]
            ctl_values = ctl_values[np.isfinite(ctl_values)]

            if len(tbi_values) < 2 or len(ctl_values) < 2:
                u_stat = np.nan
                p_value = np.nan
                effect = np.nan
                rank_biserial = np.nan
            else:
                u_stat, p_value = stats.mannwhitneyu(
                    tbi_values,
                    ctl_values,
                    alternative="two-sided",
                )
                effect = hedges_g(tbi_values, ctl_values)
                rank_biserial = rank_biserial_from_u(
                    u_stat=u_stat,
                    n_x=len(tbi_values),
                    n_y=len(ctl_values),
                )

            tbi_mean = safe_nanmean(tbi_values)
            control_mean = safe_nanmean(ctl_values)
            tbi_median = safe_nanmedian(tbi_values)
            control_median = safe_nanmedian(ctl_values)

            records.append(
                {
                    **context,
                    "feature": feature,
                    "tbi_mean": tbi_mean,
                    "control_mean": control_mean,
                    "delta_tbi_minus_control": (
                        tbi_mean - control_mean
                    ),
                    "tbi_median": tbi_median,
                    "control_median": control_median,
                    "delta_median_tbi_minus_control": (
                        tbi_median - control_median
                    ),
                    "tbi_iqr": safe_iqr(tbi_values),
                    "control_iqr": safe_iqr(ctl_values),
                    "hedges_g": effect,
                    "rank_biserial": rank_biserial,
                    "u_stat": u_stat,
                    "p_value": p_value,
                    "n_tbi": len(tbi_values),
                    "n_control": len(ctl_values),
                }
            )

    result = pd.DataFrame(records)

    if len(result) > 0:
        result["q_fdr"] = multipletests(
            result["p_value"].fillna(1.0),
            method="fdr_bh",
        )[1]

        result["significant_fdr_0_05"] = result["q_fdr"] < 0.05

    return result

In [ ]:
# @title 8.2. Статистика для ROI-спектральных признаков

roi_features = bandpower_roi.merge(
    spectral_shape_roi,
    on=["group", "subject_id", "record_id", "level", "region"],
    how="left",
)

roi_features = roi_features.merge(
    aperiodic_alpha_roi,
    on=["group", "subject_id", "record_id", "level", "region"],
    how="left",
)

feature_columns = [
    column for column in roi_features.columns
    if column not in [
        "group",
        "subject_id",
        "record_id",
        "level",
        "region",
    ]
    and pd.api.types.is_numeric_dtype(roi_features[column])
]

spectral_stats_roi = compare_groups_by_features(
    df=roi_features,
    feature_columns=feature_columns,
    context_columns=["region"],
)

spectral_stats_roi = spectral_stats_roi.sort_values(
    ["q_fdr", "region", "feature"]
)

save_table(
    spectral_stats_roi,
    OUT["tables"] / "spectral_stats_roi.csv",
)

display(spectral_stats_roi.head(30))

In [ ]:
# @title 8.3. Subject-level статистика для ROI-спектральных признаков

roi_features_subject = (
    roi_features
    .groupby(["group", "subject_id", "region"], as_index=False)
    .mean(numeric_only=True)
)

spectral_stats_roi_subject = compare_groups_by_features(
    df=roi_features_subject,
    feature_columns=feature_columns,
    context_columns=["region"],
)

spectral_stats_roi_subject = spectral_stats_roi_subject.sort_values(
    ["q_fdr", "region", "feature"]
)

save_table(
    spectral_stats_roi_subject,
    OUT["tables"] / "spectral_stats_roi_subject_level.csv",
)

display(spectral_stats_roi_subject.head(30))

In [ ]:
# @title 8.4. Heatmap Hedges' g для относительной мощности по ROI

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ---------------------------------------------------------------------
# Настройки
# ---------------------------------------------------------------------
bands_order = ["delta", "theta", "alpha", "beta", "gamma"]
roi_order = ["frontal", "central", "temporal", "parietal", "occipital"]

band_feature_map = {
    "delta": "rel_power_delta",
    "theta": "rel_power_theta",
    "alpha": "rel_power_alpha",
    "beta": "rel_power_beta",
    "gamma": "rel_power_gamma",
}

band_labels = {
    "delta": "δ",
    "theta": "θ",
    "alpha": "α",
    "beta": "β",
    "gamma": "γ",
}

roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

# ---------------------------------------------------------------------
# Берём subject-level статистику, если она есть
# ---------------------------------------------------------------------
if "spectral_stats_roi_subject" in globals():
    stats_plot = spectral_stats_roi_subject.copy()
elif "spectral_stats_roi" in globals():
    stats_plot = spectral_stats_roi.copy()
else:
    raise NameError(
        "Не найдена таблица статистики. Ожидается "
        "`spectral_stats_roi_subject` или `spectral_stats_roi`."
    )

required_columns = ["region", "feature", "hedges_g", "q_fdr"]

missing_columns = [
    col for col in required_columns
    if col not in stats_plot.columns
]

if missing_columns:
    raise ValueError(
        "В таблице статистики отсутствуют колонки: "
        f"{missing_columns}"
    )

# ---------------------------------------------------------------------
# Формируем матрицы Hedges' g и q_fdr
# ---------------------------------------------------------------------
records = []

for band in bands_order:
    feature_name = band_feature_map[band]

    for roi in roi_order:
        sub = stats_plot[
            (stats_plot["region"] == roi)
            & (stats_plot["feature"] == feature_name)
        ]

        if sub.empty:
            value_g = np.nan
            value_q = np.nan
        else:
            value_g = sub["hedges_g"].iloc[0]
            value_q = sub["q_fdr"].iloc[0]

        records.append(
            {
                "band": band,
                "region": roi,
                "hedges_g": value_g,
                "q_fdr": value_q,
            }
        )

heatmap_stats = pd.DataFrame(records)

g_pivot = (
    heatmap_stats
    .pivot(index="band", columns="region", values="hedges_g")
    .loc[bands_order, roi_order]
)

q_pivot = (
    heatmap_stats
    .pivot(index="band", columns="region", values="q_fdr")
    .loc[bands_order, roi_order]
)

abs_max = np.nanmax(np.abs(g_pivot.values))
norm = TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)

# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5.8))

im = ax.imshow(
    g_pivot.values,
    cmap="RdBu_r",
    norm=norm,
    aspect="auto",
)

ax.set_title(
    "Размер эффекта Hedges' g: относительная мощность",
    fontsize=14,
    fontweight="bold",
    pad=10,
)

ax.set_xlabel("ROI")
ax.set_ylabel("Диапазон частот")

ax.set_xticks(np.arange(len(roi_order)))
ax.set_xticklabels(
    [roi_labels[roi] for roi in roi_order],
    rotation=25,
    ha="right",
)

ax.set_yticks(np.arange(len(bands_order)))
ax.set_yticklabels([band_labels[band] for band in bands_order])

# Сетка между ячейками
ax.set_xticks(np.arange(-0.5, len(roi_order), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(bands_order), 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.0)
ax.tick_params(which="minor", bottom=False, left=False)

# Подписи: g + звёздочки FDR
for i, band in enumerate(bands_order):
    for j, roi in enumerate(roi_order):
        g_value = g_pivot.loc[band, roi]
        q_value = q_pivot.loc[band, roi]

        if pd.isna(g_value):
            label = "NA"
            text_color = "black"
        else:
            if pd.isna(q_value):
                stars = ""
            elif q_value < 0.001:
                stars = "***"
            elif q_value < 0.01:
                stars = "**"
            elif q_value < 0.05:
                stars = "*"
            else:
                stars = ""

            label = f"{g_value:.2f}{stars}"
            text_color = "white" if abs(g_value) > 0.55 * abs_max else "black"

        ax.text(
            j,
            i,
            label,
            ha="center",
            va="center",
            fontsize=10,
            fontweight="bold" if stars else "normal",
            color=text_color,
        )

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Hedges' g, ЧМТ − Контроль")

fig.subplots_adjust(
    left=0.10,
    right=0.94,
    top=0.86,
    bottom=0.18,
)

figure_path = OUT["figures"] / "statistics_hedges_g_relative_bandpower_heatmap.png"

save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 8.5. Heatmap Hedges' g для всех ROI-спектральных признаков

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# ---------------------------------------------------------------------
# Берём subject-level статистику
# ---------------------------------------------------------------------
if "spectral_stats_roi_subject" in globals():
    stats_plot = spectral_stats_roi_subject.copy()
elif "spectral_stats_roi" in globals():
    stats_plot = spectral_stats_roi.copy()
else:
    raise NameError(
        "Не найдена таблица статистики. Ожидается "
        "`spectral_stats_roi_subject` или `spectral_stats_roi`."
    )

required_columns = ["region", "feature", "hedges_g", "q_fdr"]

missing_columns = [
    col for col in required_columns
    if col not in stats_plot.columns
]

if missing_columns:
    raise ValueError(
        "В таблице статистики отсутствуют колонки: "
        f"{missing_columns}"
    )

roi_order = ["frontal", "central", "temporal", "parietal", "occipital"]

roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

feature_order = [
    "slow_alpha_ratio",
    "slow_beta_ratio",
    "log_slow_fast_ratio",
    "sef50",
    "sef95",
    "spectral_entropy",
    "spectral_flatness",
    "aperiodic_exponent",
    "aperiodic_offset",
    "alpha_peak_frequency",
    "alpha_peak_amplitude",
    "alpha_peak_residual",
]

feature_labels = {
    "slow_alpha_ratio": "(δ+θ)/α",
    "slow_beta_ratio": "(δ+θ)/β",
    "log_slow_fast_ratio": "log((δ+θ)/(α+β))",
    "sef50": "SEF50",
    "sef95": "SEF95",
    "spectral_entropy": "Entropy",
    "spectral_flatness": "Flatness",
    "aperiodic_exponent": "1/f exponent",
    "aperiodic_offset": "1/f offset",
    "alpha_peak_frequency": "α peak freq",
    "alpha_peak_amplitude": "α peak amp",
    "alpha_peak_residual": "α residual peak",
}

# Оставляем только признаки, которые реально есть в статистике
available_features = [
    feature for feature in feature_order
    if feature in set(stats_plot["feature"])
]

if not available_features:
    raise ValueError(
        "Не найдено ни одного ожидаемого признака в stats_plot['feature']."
    )

stats_plot = stats_plot[
    stats_plot["feature"].isin(available_features)
    & stats_plot["region"].isin(roi_order)
].copy()

g_pivot = (
    stats_plot
    .pivot(index="feature", columns="region", values="hedges_g")
    .reindex(index=available_features, columns=roi_order)
)

q_pivot = (
    stats_plot
    .pivot(index="feature", columns="region", values="q_fdr")
    .reindex(index=available_features, columns=roi_order)
)

abs_max = np.nanmax(np.abs(g_pivot.values))
norm = TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)

# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
fig_height = max(6, 0.48 * len(available_features) + 2)

fig, ax = plt.subplots(figsize=(9.5, fig_height))

im = ax.imshow(
    g_pivot.values,
    cmap="RdBu_r",
    norm=norm,
    aspect="auto",
)

ax.set_title(
    "Размер эффекта Hedges' g для ROI-спектральных признаков",
    fontsize=14,
    fontweight="bold",
    pad=10,
)

ax.set_xlabel("ROI")
ax.set_ylabel("Признак")

ax.set_xticks(np.arange(len(roi_order)))
ax.set_xticklabels(
    [roi_labels[roi] for roi in roi_order],
    rotation=25,
    ha="right",
)

ax.set_yticks(np.arange(len(available_features)))
ax.set_yticklabels(
    [feature_labels.get(feature, feature) for feature in available_features]
)

ax.set_xticks(np.arange(-0.5, len(roi_order), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(available_features), 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.0)
ax.tick_params(which="minor", bottom=False, left=False)

for i, feature in enumerate(available_features):
    for j, roi in enumerate(roi_order):
        g_value = g_pivot.loc[feature, roi]
        q_value = q_pivot.loc[feature, roi]

        if pd.isna(g_value):
            label = ""
            text_color = "black"
        else:
            if pd.isna(q_value):
                stars = ""
            elif q_value < 0.001:
                stars = "***"
            elif q_value < 0.01:
                stars = "**"
            elif q_value < 0.05:
                stars = "*"
            else:
                stars = ""

            label = f"{g_value:.2f}{stars}"
            text_color = "white" if abs(g_value) > 0.55 * abs_max else "black"

        ax.text(
            j,
            i,
            label,
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold" if stars else "normal",
            color=text_color,
        )

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Hedges' g, ЧМТ − Контроль")

fig.subplots_adjust(
    left=0.24,
    right=0.94,
    top=0.90,
    bottom=0.12,
)

figure_path = OUT["figures"] / "statistics_hedges_g_all_roi_features_heatmap.png"

save_figure(fig, figure_path)

plt.show()

In [ ]:
# @title 8.6. Топ ROI-признаков по размеру эффекта

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------
# Берём subject-level статистику
# ---------------------------------------------------------------------
if "spectral_stats_roi_subject" in globals():
    stats_plot = spectral_stats_roi_subject.copy()
elif "spectral_stats_roi" in globals():
    stats_plot = spectral_stats_roi.copy()
else:
    raise NameError(
        "Не найдена таблица статистики. Ожидается "
        "`spectral_stats_roi_subject` или `spectral_stats_roi`."
    )

required_columns = ["feature", "region", "hedges_g", "q_fdr"]

missing_columns = [
    col for col in required_columns
    if col not in stats_plot.columns
]

if missing_columns:
    raise ValueError(
        "В таблице статистики отсутствуют колонки: "
        f"{missing_columns}"
    )

roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

feature_labels = {
    "rel_power_delta": "δ relative power",
    "rel_power_theta": "θ relative power",
    "rel_power_alpha": "α relative power",
    "rel_power_beta": "β relative power",
    "rel_power_gamma": "γ relative power",
    "slow_alpha_ratio": "(δ+θ)/α",
    "slow_beta_ratio": "(δ+θ)/β",
    "log_slow_fast_ratio": "log((δ+θ)/(α+β))",
    "sef50": "SEF50",
    "sef95": "SEF95",
    "spectral_entropy": "Entropy",
    "spectral_flatness": "Flatness",
    "aperiodic_exponent": "1/f exponent",
    "aperiodic_offset": "1/f offset",
}

top_n = 25

top_df = (
    stats_plot
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["hedges_g"])
    .copy()
)

top_df["abs_g"] = top_df["hedges_g"].abs()

top_df = (
    top_df
    .sort_values("abs_g", ascending=False)
    .head(top_n)
    .sort_values("hedges_g")
    .copy()
)

top_df["label"] = (
    top_df["feature"].map(feature_labels).fillna(top_df["feature"])
    + " | "
    + top_df["region"].map(roi_labels).fillna(top_df["region"])
)

top_df["is_significant"] = top_df["q_fdr"] < 0.05

# ---------------------------------------------------------------------
# Фигура
# ---------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 8))

y_pos = np.arange(len(top_df))

colors = [
    COL_TBI if value > 0 else COL_CONTROL
    for value in top_df["hedges_g"]
]

markers = [
    "o" if sig else "x"
    for sig in top_df["is_significant"]
]

for idx, (_, row) in enumerate(top_df.iterrows()):
    ax.scatter(
        row["hedges_g"],
        idx,
        color=COL_TBI if row["hedges_g"] > 0 else COL_CONTROL,
        marker="o" if row["is_significant"] else "x",
        s=70 if row["is_significant"] else 55,
        linewidth=1.4,
    )

ax.axvline(
    0,
    color="black",
    linewidth=0.9,
    linestyle="--",
    alpha=0.7,
)

ax.set_yticks(y_pos)
ax.set_yticklabels(top_df["label"])

ax.set_xlabel("Hedges' g, ЧМТ − Контроль")
ax.set_title(
    f"Топ-{top_n} ROI-признаков по абсолютному размеру эффекта",
    fontsize=14,
    fontweight="bold",
    pad=10,
)

ax.grid(axis="x", linestyle="--", alpha=0.25)
ax.grid(axis="y", linestyle="", alpha=0)

# Подписи q_fdr
for idx, (_, row) in enumerate(top_df.iterrows()):
    ax.text(
        row["hedges_g"],
        idx,
        f"  q={row['q_fdr']:.3g}",
        va="center",
        ha="left" if row["hedges_g"] >= 0 else "right",
        fontsize=8.5,
        color="dimgray",
    )

fig.subplots_adjust(
    left=0.36,
    right=0.96,
    top=0.90,
    bottom=0.10,
)

figure_path = OUT["figures"] / "statistics_top_roi_features_by_hedges_g.png"

save_figure(fig, figure_path)

plt.show()

# 9. Итоговая таблица спектральных признаков

В этом разделе все спектральные признаки объединяются в широкую таблицу на уровне записи.

Формат имён признаков:

```text
<block>__<region>__<metric>
```

Например:

```text
bp__frontal__rel_power_delta
spec_shape__occipital__sef95
aper__central__aperiodic_exponent
alpha_peak__parietal__alpha_peak_frequency
```

Такая структура удобна для последующего статистического анализа и машинного обучения.

In [ ]:
# @title 9.1. Формирование wide-таблицы спектральных признаков

def make_wide_features(
    df: pd.DataFrame,
    id_columns: list[str],
    region_column: str,
    feature_columns: list[str],
    block_name: str,
) -> pd.DataFrame:
    """
    Преобразует long ROI-таблицу в wide-формат.

    Parameters
    ----------
    df : pd.DataFrame
        Long-таблица признаков.
    id_columns : list[str]
        Колонки идентификаторов.
    region_column : str
        Колонка региона.
    feature_columns : list[str]
        Признаки.
    block_name : str
        Название блока признаков.

    Returns
    -------
    pd.DataFrame
        Wide-таблица.
    """
    long_records = []

    for _, row in df.iterrows():
        base = {column: row[column] for column in id_columns}
        region = row[region_column]

        for feature in feature_columns:
            long_records.append(
                {
                    **base,
                    "feature_name": f"{block_name}__{region}__{feature}",
                    "feature_value": row[feature],
                }
            )

    long_df = pd.DataFrame(long_records)

    wide_df = long_df.pivot_table(
        index=id_columns,
        columns="feature_name",
        values="feature_value",
        aggfunc="first",
    ).reset_index()

    wide_df.columns.name = None

    return wide_df


id_columns = ["group", "subject_id", "record_id"]

bp_columns = [
    column for column in bandpower_roi.columns
    if (
        column.startswith("abs_power_")
        or column.startswith("rel_power_")
        or column.startswith("idx_")
        or column == "total_power_1_45"
    )
]

shape_columns = [
    "sef50",
    "sef95",
    "spectral_entropy",
    "spectral_flatness",
]

aper_alpha_columns = [
    "aperiodic_exponent",
    "aperiodic_offset",
    "alpha_peak_frequency",
    "alpha_peak_amplitude",
    "alpha_peak_residual_amplitude",
]

bp_wide = make_wide_features(
    df=bandpower_roi,
    id_columns=id_columns,
    region_column="region",
    feature_columns=bp_columns,
    block_name="bp",
)

shape_wide = make_wide_features(
    df=spectral_shape_roi,
    id_columns=id_columns,
    region_column="region",
    feature_columns=shape_columns,
    block_name="spec_shape",
)

aper_alpha_wide = make_wide_features(
    df=aperiodic_alpha_roi,
    id_columns=id_columns,
    region_column="region",
    feature_columns=aper_alpha_columns,
    block_name="aper_alpha",
)

spectral_features_record_level = (
    bp_wide
    .merge(shape_wide, on=id_columns, how="outer")
    .merge(aper_alpha_wide, on=id_columns, how="outer")
)

save_table(
    spectral_features_record_level,
    OUT["tables"] / "spectral_features_record_level.csv",
)

print("Итоговая таблица спектральных признаков:")
print(spectral_features_record_level.shape)
display(spectral_features_record_level.head())

In [ ]:
# @title 9.2. Карта признаков и пропуски

feature_columns = [
    column for column in spectral_features_record_level.columns
    if column not in ["group", "subject_id", "record_id"]
]

feature_map = pd.DataFrame(
    {
        "feature": feature_columns,
        "block": [
            feature.split("__")[0]
            if "__" in feature else "unknown"
            for feature in feature_columns
        ],
        "region": [
            feature.split("__")[1]
            if "__" in feature and len(feature.split("__")) > 2
            else "unknown"
            for feature in feature_columns
        ],
    }
)

missing_summary = pd.DataFrame(
    {
        "feature": feature_columns,
        "missing_prop": spectral_features_record_level[
            feature_columns
        ].isna().mean().values,
        "zero_variance": [
            spectral_features_record_level[column].nunique(dropna=True) <= 1
            for column in feature_columns
        ],
    }
)

save_table(
    feature_map,
    OUT["tables"] / "spectral_feature_map.csv",
)

save_table(
    missing_summary,
    OUT["tables"] / "spectral_feature_missingness.csv",
)

display(missing_summary.sort_values("missing_prop", ascending=False).head(20))

# 10. Финальная проверка результатов

На последнем этапе проверяется наличие ключевых выходных файлов.

Основные результаты ноутбука:

- `psd_global_long.csv`;
- `psd_roi_long.csv`;
- `bandpower_roi.csv`;
- `spectral_shape_roi.csv`;
- `aperiodic_alpha_roi.csv`;
- `residual_roi_long.csv`;
- `spectral_stats_roi.csv`;
- `spectral_features_record_level.csv`;
- `spectral_feature_map.csv`;
- `spectral_feature_missingness.csv`.

In [ ]:
# @title 10.1. Финальная проверка выходных файлов

required_outputs = {
    "psd_global_long": OUT["tables"] / "psd_global_long.csv",
    "psd_roi_long": OUT["tables"] / "psd_roi_long.csv",
    "bandpower_global": OUT["tables"] / "bandpower_global.csv",
    "bandpower_roi": OUT["tables"] / "bandpower_roi.csv",
    "spectral_shape_global": OUT["tables"] / "spectral_shape_global.csv",
    "spectral_shape_roi": OUT["tables"] / "spectral_shape_roi.csv",
    "aperiodic_alpha_global": OUT["tables"] / "aperiodic_alpha_global.csv",
    "aperiodic_alpha_roi": OUT["tables"] / "aperiodic_alpha_roi.csv",
    "residual_roi_long": OUT["tables"] / "residual_roi_long.csv",
    "spectral_stats_roi": OUT["tables"] / "spectral_stats_roi.csv",
    "spectral_features_record_level": (
        OUT["tables"] / "spectral_features_record_level.csv"
    ),
    "spectral_feature_map": OUT["tables"] / "spectral_feature_map.csv",
    "spectral_feature_missingness": (
        OUT["tables"] / "spectral_feature_missingness.csv"
    ),
}

print("Проверка ключевых выходных файлов")
print("-" * 70)

for name, path in required_outputs.items():
    status = "OK" if path.exists() else "НЕ НАЙДЕН"
    print(f"{name}: {status}")
    print(f"  {path}")

print("\nСпектральный анализ завершён.")

# Итог ноутбука

В результате выполнения ноутбука рассчитаны спектральные характеристики ЭЭГ на уровне записи.

Получены следующие группы признаков:

1. абсолютная и относительная диапазонная мощность;
2. индексы slow/fast;
3. SEF50 и SEF95;
4. спектральная энтропия;
5. spectral flatness;
6. alpha peak frequency и amplitude;
7. апериодический exponent и offset;
8. alpha peak над апериодическим фоном.

Основной выходной файл:

```text
spectral_features_record_level.csv
```

Эта таблица может быть использована отдельно или объединена с временными, сетевыми и событийными признаками в следующем ноутбуке.

# 11. Channel-level bandpower и топографическая статистика

Этот дополнительный раздел - пространственный анализ спектральных эффектов на уровне отдельных каналов.

Задачи блока:

1. рассчитать bandpower для каждого канала;
2. сравнить группы ЧМТ и контроля по каждому каналу и диапазону;
3. выполнить FDR-коррекцию по каналам внутри каждого диапазона;
4. сохранить таблицу channel-level статистики;
5. построить топографические карты групповых различий.

Этот блок нужен для ВКР, если в тексте используются топографические выводы о δ↑, β↓ или γ↓.

In [ ]:
# @title 11.1. Channel-level bandpower

def compute_channel_bandpower_features(
    psd_cache_records: list[dict],
    config: dict,
) -> pd.DataFrame:
    """
    Рассчитывает bandpower на уровне отдельных каналов.

    Parameters
    ----------
    psd_cache_records : list[dict]
        Список записей с PSD на уровне каналов.
    config : dict
        Конфигурация анализа.

    Returns
    -------
    pd.DataFrame
        Channel-level таблица bandpower.
    """
    records = []

    for item in psd_cache_records:
        freqs = item["freqs"]
        psd = item["psd"]
        ch_names = item["ch_names"]

        total_power = np.array(
            [
                integrate_band_power(
                    freqs=freqs,
                    psd_values=psd[channel_index, :],
                    band=(config["fmin"], config["fmax"]),
                )
                for channel_index in range(len(ch_names))
            ]
        )

        for channel_index, channel in enumerate(ch_names):
            base = {
                "group": item["group"],
                "subject_id": item["subject_id"],
                "record_id": item["record_id"],
                "channel": channel,
                "total_power_1_45": total_power[channel_index],
            }

            for band_name, band_range in config["bands"].items():
                abs_power = integrate_band_power(
                    freqs=freqs,
                    psd_values=psd[channel_index, :],
                    band=band_range,
                )

                rel_power = (
                    abs_power / total_power[channel_index]
                    if total_power[channel_index] > 0
                    else np.nan
                )

                records.append(
                    {
                        **base,
                        "band": band_name,
                        "abs_power": abs_power,
                        "rel_power": rel_power,
                    }
                )

    return pd.DataFrame(records)


# Повторно рассчитываем PSD на уровне каналов в компактный cache-список.
channel_psd_cache = []

for _, row in tqdm(
    analysis_inventory.iterrows(),
    total=len(analysis_inventory),
    desc="Channel-level PSD cache",
):
    try:
        epochs = load_epochs_file(row["common_epochs_path"])
        psd, freqs, ch_names = compute_record_psd(
            epochs=epochs,
            config=CONFIG,
        )

        channel_psd_cache.append(
            {
                "group": row["group"],
                "subject_id": row["subject_id"],
                "record_id": row["record_id"],
                "psd": psd,
                "freqs": freqs,
                "ch_names": ch_names,
            }
        )

        del epochs
        gc.collect()

    except Exception as error:
        logger.exception(
            "Ошибка channel-level PSD для записи %s: %s",
            row.get("record_id"),
            error,
        )

channel_bandpower_long = compute_channel_bandpower_features(
    psd_cache_records=channel_psd_cache,
    config=CONFIG,
)

save_table(
    channel_bandpower_long,
    OUT["tables"] / "channel_bandpower_long.csv",
)

display(channel_bandpower_long.head())

In [ ]:
# @title 11.1.1. Subject-level агрегация channel-level bandpower

channel_bandpower_subject = (
    channel_bandpower_long
    .groupby(["group", "subject_id", "channel", "band"], as_index=False)
    .mean(numeric_only=True)
)

save_table(
    channel_bandpower_subject,
    OUT["tables"] / "channel_bandpower_subject_level.csv",
)

display(channel_bandpower_subject.head())

In [ ]:
# @title 11.2. Channel-level статистика с FDR-коррекцией

def compare_channel_bandpower(
    df: pd.DataFrame,
    value_column: str = "rel_power",
) -> pd.DataFrame:
    """
    Сравнивает группы по channel-level bandpower.

    Parameters
    ----------
    df : pd.DataFrame
        Channel-level таблица.
    value_column : str
        Название анализируемого показателя.

    Returns
    -------
    pd.DataFrame
        Таблица статистики по каналам и диапазонам.
    """
    records = []

    for (band, channel), sub_df in df.groupby(["band", "channel"]):
        tbi_values = sub_df[sub_df["group"] == "TBI"][value_column].to_numpy()
        ctl_values = sub_df[
            sub_df["group"] == "Control"
        ][value_column].to_numpy()

        tbi_values = tbi_values[np.isfinite(tbi_values)]
        ctl_values = ctl_values[np.isfinite(ctl_values)]

        if len(tbi_values) < 2 or len(ctl_values) < 2:
            p_value = np.nan
            effect = np.nan
        else:
            _, p_value = stats.mannwhitneyu(
                tbi_values,
                ctl_values,
                alternative="two-sided",
            )
            effect = hedges_g(tbi_values, ctl_values)

        records.append(
            {
                "band": band,
                "channel": channel,
                "tbi_mean": np.nanmean(tbi_values),
                "control_mean": np.nanmean(ctl_values),
                "delta_tbi_minus_control": (
                    np.nanmean(tbi_values) - np.nanmean(ctl_values)
                ),
                "hedges_g": effect,
                "p_value": p_value,
                "n_tbi": len(tbi_values),
                "n_control": len(ctl_values),
            }
        )

    stats_df = pd.DataFrame(records)

    # FDR отдельно внутри каждого частотного диапазона.
    stats_df["q_fdr"] = np.nan

    for band, idx in stats_df.groupby("band").groups.items():
        p_values = stats_df.loc[idx, "p_value"].fillna(1.0).to_numpy()
        q_values = multipletests(p_values, method="fdr_bh")[1]
        stats_df.loc[idx, "q_fdr"] = q_values

    return stats_df


channel_bandpower_stats = compare_channel_bandpower(
    channel_bandpower_subject,
    value_column="rel_power",
)

save_table(
    channel_bandpower_stats,
    OUT["tables"] / "channel_bandpower_stats_fdr.csv",
)

display(
    channel_bandpower_stats
    .sort_values(["q_fdr", "band", "channel"])
    .head(30)
)

In [ ]:
# @title 11.3. Единая топографическая карта различий по диапазонам

import numpy as np
import matplotlib.pyplot as plt
import mne
from matplotlib.colors import TwoSlopeNorm


def plot_combined_channel_topomaps(
    stats_df: pd.DataFrame,
    bands: list[str],
    value_column: str,
    output_path: Path,
    title: str = "Channel-level размер эффекта по диапазонам",
) -> None:
    """
    Строит единую фигуру topomap для нескольких частотных диапазонов.

    Parameters
    ----------
    stats_df : pd.DataFrame
        Таблица channel-level статистики.
    bands : list[str]
        Список частотных диапазонов для отображения.
    value_column : str
        Колонка для отображения, например hedges_g.
    output_path : Path
        Путь для сохранения рисунка.
    title : str
        Общий заголовок фигуры.
    """
    montage = mne.channels.make_standard_montage("standard_1020")
    montage_channels = set(montage.ch_names)

    band_labels = {
        "delta": "δ\n1–4 Гц",
        "theta": "θ\n4–8 Гц",
        "alpha": "α\n8–13 Гц",
        "beta": "β\n13–30 Гц",
        "gamma": "γ\n30–45 Гц",
    }

    # -----------------------------------------------------------------
    # Собираем данные для каждого диапазона
    # -----------------------------------------------------------------
    topomap_data = {}

    for band in bands:
        band_df = stats_df[stats_df["band"] == band].copy()

        if band_df.empty:
            print(f"Нет данных для диапазона: {band}")
            continue

        band_df = band_df[
            band_df["channel"].isin(montage_channels)
        ].copy()

        band_df = band_df.dropna(subset=[value_column])

        channels = band_df["channel"].astype(str).tolist()
        values = band_df[value_column].to_numpy(dtype=float)

        if len(channels) < 3:
            print(f"Недостаточно каналов для topomap: {band}")
            continue

        info = mne.create_info(
            ch_names=channels,
            sfreq=500,
            ch_types="eeg",
        )
        info.set_montage(montage, on_missing="ignore")

        topomap_data[band] = {
            "info": info,
            "values": values,
            "channels": channels,
        }

    if not topomap_data:
        raise ValueError("Не удалось собрать данные ни для одного диапазона.")

    # -----------------------------------------------------------------
    # Общая симметричная шкала цвета для всех диапазонов
    # -----------------------------------------------------------------
    all_values = np.concatenate(
        [
            item["values"]
            for item in topomap_data.values()
        ]
    )

    vmax = np.nanmax(np.abs(all_values))

    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0

    norm = TwoSlopeNorm(
        vmin=-vmax,
        vcenter=0.0,
        vmax=vmax,
    )

    # -----------------------------------------------------------------
    # Фигура
    # -----------------------------------------------------------------
    n_bands = len(bands)

    fig, axes = plt.subplots(
        1,
        n_bands,
        figsize=(3.1 * n_bands, 3.8),
        constrained_layout=False,
    )

    axes = np.atleast_1d(axes)

    fig.suptitle(
        title,
        fontsize=14,
        fontweight="bold",
        y=0.98,
    )

    im = None

    for ax, band in zip(axes, bands):
        if band not in topomap_data:
            ax.axis("off")
            ax.set_title(
                band_labels.get(band, band),
                fontsize=11,
                fontweight="bold",
            )
            continue

        values = topomap_data[band]["values"]
        info = topomap_data[band]["info"]

        im, _ = mne.viz.plot_topomap(
            data=values,
            pos=info,
            axes=ax,
            show=False,
            contours=0,
            cmap="RdBu_r",
            vlim=(-vmax, vmax),
            names=None,
            sensors=True,
        )

        ax.set_title(
            band_labels.get(band, band),
            fontsize=11,
            fontweight="bold",
            pad=6,
        )

    # -----------------------------------------------------------------
    # Общая colorbar
    # -----------------------------------------------------------------
    cbar_ax = fig.add_axes([0.92, 0.22, 0.015, 0.55])

    cbar = fig.colorbar(
        im,
        cax=cbar_ax,
    )

    cbar.set_label(
        "Hedges' g\nЧМТ − Контроль",
        fontsize=10,
    )

    # -----------------------------------------------------------------
    # Подпись направления эффекта
    # -----------------------------------------------------------------
    fig.text(
        0.5,
        0.05,
        "Положительные значения: выше у ЧМТ; отрицательные значения: выше у контроля",
        ha="center",
        va="center",
        fontsize=10,
        color="dimgray",
    )

    fig.subplots_adjust(
        left=0.03,
        right=0.90,
        top=0.82,
        bottom=0.13,
        wspace=0.08,
    )

    save_figure(fig, output_path)

    plt.show()


plot_combined_channel_topomaps(
    stats_df=channel_bandpower_stats,
    bands=list(CONFIG["bands"].keys()),
    value_column="hedges_g",
    output_path=OUT["figures"] / "topomap_hedges_g_all_bands_combined.png",
    title="Топографическое распределение размера эффекта по диапазонам",
)

### Результат channel-level анализа

В этом блоке сохранены:

```text
channel_bandpower_long.csv
channel_bandpower_stats_fdr.csv
topomap_hedges_g_<band>.png
```

Эти результаты можно использовать для иллюстрации пространственного распределения спектральных эффектов.

In [ ]:
# @title 12.3. Итоговый отчёт по спектральным признакам и статистике

from pathlib import Path
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# 1. Выбор основной статистической таблицы
# ---------------------------------------------------------------------
# Для финальной интерпретации используем subject-level статистику.
# Если она есть в окружении, берём её. Если нет — явно сообщаем об ошибке.

if "spectral_stats_roi_subject" in globals():
    final_stats = spectral_stats_roi_subject.copy()
    stats_level = "subject-level"
elif "spectral_stats_roi" in globals():
    final_stats = spectral_stats_roi.copy()
    stats_level = "record-level"
    print(
        "ВНИМАНИЕ: используется record-level статистика. "
        "Для финального отчёта лучше использовать spectral_stats_roi_subject."
    )
else:
    raise NameError(
        "Не найдена таблица статистики. Ожидается "
        "`spectral_stats_roi_subject` или `spectral_stats_roi`."
    )


# ---------------------------------------------------------------------
# 2. Словари для красивых подписей
# ---------------------------------------------------------------------
roi_labels = {
    "frontal": "Лобная",
    "central": "Центральная",
    "temporal": "Височная",
    "parietal": "Теменная",
    "occipital": "Затылочная",
}

feature_labels = {
    # Абсолютная мощность
    "abs_power_delta": "Абсолютная мощность δ",
    "abs_power_theta": "Абсолютная мощность θ",
    "abs_power_alpha": "Абсолютная мощность α",
    "abs_power_beta": "Абсолютная мощность β",
    "abs_power_gamma": "Абсолютная мощность γ",

    # Относительная мощность
    "rel_power_delta": "Относительная мощность δ",
    "rel_power_theta": "Относительная мощность θ",
    "rel_power_alpha": "Относительная мощность α",
    "rel_power_beta": "Относительная мощность β",
    "rel_power_gamma": "Относительная мощность γ",

    # Total power
    "total_power_1_45": "Суммарная мощность 1–45 Гц",

    # Индексы
    "idx_slow_alpha": "(δ + θ) / α",
    "idx_slow_beta": "(δ + θ) / β",
    "idx_log_slow_fast": "log((δ + θ) / (α + β))",
    "slow_alpha_ratio": "(δ + θ) / α",
    "slow_beta_ratio": "(δ + θ) / β",
    "log_slow_fast_ratio": "log((δ + θ) / (α + β))",

    # Форма спектра
    "sef50": "SEF50",
    "sef95": "SEF95",
    "spectral_entropy": "Спектральная энтропия",
    "spectral_flatness": "Спектральная плоскостность",

    # 1/f
    "aperiodic_exponent": "Апериодический показатель 1/f",
    "aperiodic_offset": "Апериодический offset",

    # Alpha peak
    "alpha_peak_frequency": "Частота α-пика",
    "alpha_peak_freq": "Частота α-пика",
    "alpha_peak_amplitude": "Амплитуда α-пика",
    "alpha_peak_residual_amplitude": "Амплитуда α-пика над 1/f",
    "alpha_peak_residual": "Амплитуда α-пика над 1/f",
}

feature_family_map = {
    "abs_power": "Абсолютная мощность",
    "rel_power": "Относительная мощность",
    "total_power": "Суммарная мощность",
    "idx_": "Индексы замедления",
    "slow_": "Индексы замедления",
    "log_slow": "Индексы замедления",
    "sef": "Форма спектра",
    "spectral_entropy": "Форма спектра",
    "spectral_flatness": "Форма спектра",
    "aperiodic": "Апериодическая компонента",
    "alpha_peak": "Alpha peak",
}


def get_feature_family(feature_name: str) -> str:
    """Определяет семейство признака по названию."""
    feature_name = str(feature_name)

    for key, family in feature_family_map.items():
        if feature_name.startswith(key) or key in feature_name:
            return family

    return "Другие признаки"


def p_to_stars(q_value: float) -> str:
    """Преобразует q-FDR в звёздочки значимости."""
    if pd.isna(q_value):
        return ""

    if q_value < 0.001:
        return "***"

    if q_value < 0.01:
        return "**"

    if q_value < 0.05:
        return "*"

    return ""


def format_float(value, digits: int = 3) -> str:
    """Форматирует числа для отчёта."""
    if pd.isna(value):
        return "NA"

    value = float(value)

    if value == 0:
        return "0"

    if abs(value) < 0.001 or abs(value) >= 10000:
        return f"{value:.3e}"

    return f"{value:.{digits}f}"


def format_median_iqr(median, iqr) -> str:
    """Форматирует медиану и IQR."""
    return f"{format_float(median)} [{format_float(iqr)}]"


def get_direction(delta_value: float) -> str:
    """Возвращает направление эффекта."""
    if pd.isna(delta_value):
        return "NA"

    if delta_value > 0:
        return "выше при ЧМТ"

    if delta_value < 0:
        return "ниже при ЧМТ"

    return "без разницы"


# ---------------------------------------------------------------------
# 3. Проверка обязательных колонок
# ---------------------------------------------------------------------
required_columns = [
    "region",
    "feature",
    "tbi_mean",
    "control_mean",
    "delta_tbi_minus_control",
    "tbi_median",
    "control_median",
    "tbi_iqr",
    "control_iqr",
    "hedges_g",
    "rank_biserial",
    "p_value",
    "q_fdr",
    "n_tbi",
    "n_control",
]

missing_columns = [
    col for col in required_columns
    if col not in final_stats.columns
]

if missing_columns:
    raise ValueError(
        "В таблице статистики отсутствуют обязательные колонки: "
        f"{missing_columns}"
    )


# ---------------------------------------------------------------------
# 4. Формирование полной отчётной таблицы
# ---------------------------------------------------------------------
report_df = final_stats.copy()

report_df["roi_ru"] = report_df["region"].map(roi_labels).fillna(
    report_df["region"]
)

report_df["feature_ru"] = report_df["feature"].map(feature_labels).fillna(
    report_df["feature"]
)

report_df["feature_family"] = report_df["feature"].map(get_feature_family)

report_df["direction"] = report_df["delta_median_tbi_minus_control"].map(
    get_direction
)

report_df["significance"] = report_df["q_fdr"].map(p_to_stars)

report_df["tbi_median_iqr"] = report_df.apply(
    lambda row: format_median_iqr(row["tbi_median"], row["tbi_iqr"]),
    axis=1,
)

report_df["control_median_iqr"] = report_df.apply(
    lambda row: format_median_iqr(row["control_median"], row["control_iqr"]),
    axis=1,
)

report_df["abs_hedges_g"] = report_df["hedges_g"].abs()

report_df["is_significant_fdr_0_05"] = report_df["q_fdr"] < 0.05

# Удобный порядок колонок.
report_columns = [
    "feature_family",
    "roi_ru",
    "region",
    "feature_ru",
    "feature",
    "tbi_median_iqr",
    "control_median_iqr",
    "tbi_mean",
    "control_mean",
    "delta_tbi_minus_control",
    "delta_median_tbi_minus_control",
    "direction",
    "hedges_g",
    "rank_biserial",
    "p_value",
    "q_fdr",
    "significance",
    "is_significant_fdr_0_05",
    "n_tbi",
    "n_control",
]

report_df = report_df[report_columns].sort_values(
    ["q_fdr", "feature_family", "roi_ru", "feature_ru"]
)


# ---------------------------------------------------------------------
# 5. Значимые результаты и топ эффектов
# ---------------------------------------------------------------------
significant_report_df = report_df[
    report_df["is_significant_fdr_0_05"]
].copy()

top_effects_df = (
    report_df
    .dropna(subset=["hedges_g"])
    .sort_values("hedges_g", key=lambda x: x.abs(), ascending=False)
    .head(30)
    .copy()
)

family_summary = (
    report_df
    .groupby("feature_family", as_index=False)
    .agg(
        n_tests=("feature", "count"),
        n_significant_fdr_0_05=("is_significant_fdr_0_05", "sum"),
        max_abs_hedges_g=("hedges_g", lambda x: np.nanmax(np.abs(x))),
        min_q_fdr=("q_fdr", "min"),
    )
    .sort_values("min_q_fdr")
)


# ---------------------------------------------------------------------
# 6. Сохранение таблиц
# ---------------------------------------------------------------------
report_dir = OUT["tables"] / "final_report"
report_dir.mkdir(parents=True, exist_ok=True)

full_report_path = report_dir / "spectral_final_report_all_features.csv"
significant_report_path = report_dir / "spectral_final_report_significant_features.csv"
top_effects_path = report_dir / "spectral_final_report_top_effects.csv"
family_summary_path = report_dir / "spectral_final_report_family_summary.csv"

report_df.to_csv(full_report_path, index=False)
significant_report_df.to_csv(significant_report_path, index=False)
top_effects_df.to_csv(top_effects_path, index=False)
family_summary.to_csv(family_summary_path, index=False)


# ---------------------------------------------------------------------
# 7. Markdown-отчёт
# ---------------------------------------------------------------------
n_total = len(report_df)
n_significant = int(report_df["is_significant_fdr_0_05"].sum())

n_tbi_unique = int(report_df["n_tbi"].median())
n_control_unique = int(report_df["n_control"].median())

markdown_lines = []

markdown_lines.append("# Итоговый отчёт по спектральным признакам ЭЭГ")
markdown_lines.append("")
markdown_lines.append("## Уровень статистического анализа")
markdown_lines.append("")
markdown_lines.append(f"Основная статистическая таблица: **{stats_level}**.")
markdown_lines.append("")
markdown_lines.append(f"Число наблюдений в группе ЧМТ: **{n_tbi_unique}**.")
markdown_lines.append(f"Число наблюдений в контрольной группе: **{n_control_unique}**.")
markdown_lines.append("")
markdown_lines.append(
    "Для межгруппового сравнения использовался Mann–Whitney U test. "
    "Размер эффекта оценивался с помощью Hedges' g и rank-biserial correlation. "
    "Коррекция на множественные сравнения выполнялась методом Benjamini–Hochberg FDR."
)
markdown_lines.append("")
markdown_lines.append("## Общая сводка")
markdown_lines.append("")
markdown_lines.append(f"Всего проверено признаков/ROI-комбинаций: **{n_total}**.")
markdown_lines.append(
    f"Значимых после FDR-коррекции q < 0.05: **{n_significant}**."
)
markdown_lines.append("")

markdown_lines.append("## Сводка по семействам признаков")
markdown_lines.append("")
markdown_lines.append(
    "| Семейство признаков | Число тестов | Значимых q<0.05 | "
    "Макс. |Hedges' g| | Мин. q-FDR |"
)
markdown_lines.append(
    "|---|---:|---:|---:|---:|"
)

for _, row in family_summary.iterrows():
    markdown_lines.append(
        f"| {row['feature_family']} "
        f"| {int(row['n_tests'])} "
        f"| {int(row['n_significant_fdr_0_05'])} "
        f"| {format_float(row['max_abs_hedges_g'])} "
        f"| {format_float(row['min_q_fdr'])} |"
    )

markdown_lines.append("")
markdown_lines.append("## Топ-30 признаков по абсолютному размеру эффекта")
markdown_lines.append("")
markdown_lines.append(
    "| ROI | Признак | ЧМТ, медиана [IQR] | Контроль, медиана [IQR] | "
    "Направление | Hedges' g | q-FDR |"
)
markdown_lines.append(
    "|---|---|---:|---:|---|---:|---:|"
)

for _, row in top_effects_df.iterrows():
    markdown_lines.append(
        f"| {row['roi_ru']} "
        f"| {row['feature_ru']} "
        f"| {row['tbi_median_iqr']} "
        f"| {row['control_median_iqr']} "
        f"| {row['direction']} "
        f"| {format_float(row['hedges_g'])} "
        f"| {format_float(row['q_fdr'])}{row['significance']} |"
    )

markdown_lines.append("")
markdown_lines.append("## Значимые признаки после FDR-коррекции")
markdown_lines.append("")

if significant_report_df.empty:
    markdown_lines.append("Значимых признаков после FDR-коррекции не обнаружено.")
else:
    markdown_lines.append(
        "| Семейство | ROI | Признак | ЧМТ, медиана [IQR] | "
        "Контроль, медиана [IQR] | Направление | Hedges' g | q-FDR |"
    )
    markdown_lines.append(
        "|---|---|---|---:|---:|---|---:|---:|"
    )

    for _, row in significant_report_df.iterrows():
        markdown_lines.append(
            f"| {row['feature_family']} "
            f"| {row['roi_ru']} "
            f"| {row['feature_ru']} "
            f"| {row['tbi_median_iqr']} "
            f"| {row['control_median_iqr']} "
            f"| {row['direction']} "
            f"| {format_float(row['hedges_g'])} "
            f"| {format_float(row['q_fdr'])}{row['significance']} |"
        )

markdown_lines.append("")
markdown_lines.append("## Примечание")
markdown_lines.append("")
markdown_lines.append(
    "Положительное значение Hedges' g и положительная разность медиан "
    "означают, что признак выше в группе ЧМТ. "
    "Отрицательное значение означает, что признак ниже в группе ЧМТ "
    "по сравнению с контролем."
)
markdown_lines.append("")
markdown_lines.append(
    "Если в визуализациях использовалось ограничение оси Y по перцентилю, "
    "это касалось только отображения графиков. Статистический анализ "
    "выполнялся на полном наборе значений."
)

markdown_report_text = "\n".join(markdown_lines)

markdown_report_path = report_dir / "spectral_final_report.md"
markdown_report_path.write_text(markdown_report_text, encoding="utf-8")


# ---------------------------------------------------------------------
# 8. Вывод в ноутбуке
# ---------------------------------------------------------------------
print("Итоговый отчёт сформирован.")
print()
print(f"Уровень статистики: {stats_level}")
print(f"Всего тестов: {n_total}")
print(f"Значимых после FDR q<0.05: {n_significant}")
print()
print("Сохранённые файлы:")
print(f"- {full_report_path}")
print(f"- {significant_report_path}")
print(f"- {top_effects_path}")
print(f"- {family_summary_path}")
print(f"- {markdown_report_path}")

print("\nСводка по семействам признаков:")
display(family_summary)

print("\nТоп-30 признаков по размеру эффекта:")
display(top_effects_df)

print("\nПервые строки полной отчётной таблицы:")
display(report_df.head(30))